**LIVE RUN, redacted, 2026-11-20**

**SAMPLE RUN**

# Medical-device quality KPIs: Trinity Medical implant cell

**The story.** Trinity Medical machines Ti-6Al-4V femoral hip stems on Spindle 3 of the CNC-1
5-axis mill (Warsaw, IN). The line runs MILL, CMM INSPECT, CLEAN / PASSIVATE, STERILE PACK.
60 °C is a *validated process parameter* (OQ/PQ): parts cut while Spindle 3 is above it are
outside the validation, so they are quarantined, measured on the CMM and dispositioned (NCR).
Every reading streams into Databricks in real time and every operator decision is a signed,
Part 11-style record in the app.

This notebook turns that live data into two views:

* **EXECUTIVE** for **Alex Morgan, VP Quality & Site Head**: cost of poor quality avoided, lots and
  parts quarantined versus released, and exposure remaining, from a parameterised value model
  `events x capture rate x (lot cost - inspection cost)`.
* **OPERATIONS** for **Shreya, Head of Manufacturing Operations**: time to detect, time to
  acknowledge, NCRs and CMM inspections triggered, projected right-first-time (the app's
  first-pass-yield proxy), MTTR and the streaming
  latency distribution.

**Read-only.** Every statement is a single `SELECT` checked by an allow-list guard; Lakebase
sessions are read-only transactions. Lot numbers, NCR ids and costs are demo fixtures and
assumptions, not records of a quality system. Set `sample=true` to run anywhere from synthetic
data (bannered **SAMPLE DATA**). Pipeline operations are in `ops/notebooks/pipeline_health_check.ipynb`.

The widgets `app_backend_project` / `pipelines_project` default to this fork's `pharma-twin-*`
Lakebase projects; they are never discovered by prefix, so the notebook cannot read another
deployment's projects in the same workspace.

In [ ]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass

INDUSTRY = "manufacturing"  # internal id of the medical-device story (docs/domain-medtech.md)
MODES = ("realtime", "microbatch", "zerobus")
IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
_DBUTILS = globals().get("dbutils")  # only defined inside a Databricks notebook
_ENV_PREFIX = "MEDTECH_"
_LOOKBACK_MIN_H, _LOOKBACK_MAX_H = 1, 168
_IDENT_RE = re.compile(r"^[A-Za-z0-9_-]{0,128}$")
_PROJECT_RE = re.compile(r"^[a-z][a-z0-9-]{0,62}$")
# This fork deploys every Lakebase project under the pharma-twin prefix; the upstream app's
# iot-* projects live in the same workspace and must never be read by accident.
DEFAULT_APP_BACKEND_PROJECT = "pharma-twin-backend"
DEFAULT_PIPELINES_PROJECT = "pharma-twin-pipelines"

# ── Value model defaults (illustrative assumptions, override with the widgets) ──────────
DEFAULT_LOT_COST_USD = 24_000.0  # one lot of 15 machined Ti-6Al-4V hip stems at risk
DEFAULT_INSPECTION_COST_USD = 1_500.0  # CMM inspection + QA review to disposition one lot
DEFAULT_CAPTURE_RATE = 0.8  # share of excursions that would otherwise escape containment


@dataclass(frozen=True)
class Params:
    """Validated notebook parameters."""

    setup_id: str
    mode: str
    lookback_hours: int
    redact: bool
    sample: bool
    app_backend_project: str
    pipelines_project: str
    lot_cost: float
    inspection_cost: float
    capture_rate: float


def _declare_widgets() -> None:
    """Create the widgets (no-op outside Databricks)."""
    if _DBUTILS is None:
        return
    w = _DBUTILS.widgets
    w.text("setup_id", "", "Setup id (empty = latest medical-device setup)")
    w.dropdown("mode", MODES[0], list(MODES), "Pipeline mode")
    w.text("lookback_hours", "24", "Lookback (hours, 1-168)")
    w.dropdown("redact", "true", ["true", "false"], "Redact identifiers")
    w.dropdown("sample", "false", ["true", "false"], "Sample data (no workspace access)")
    w.text("app_backend_project", DEFAULT_APP_BACKEND_PROJECT, "App control-plane Lakebase project")
    w.text("pipelines_project", DEFAULT_PIPELINES_PROJECT, "Pipeline Lakebase project (hot tier)")
    w.text("lot_cost_usd", str(DEFAULT_LOT_COST_USD), "Value model: cost of one lot (USD)")
    w.text("inspection_cost_usd", str(DEFAULT_INSPECTION_COST_USD), "Value model: inspection (USD)")
    w.text("capture_rate", str(DEFAULT_CAPTURE_RATE), "Value model: capture rate (0-1)")


def _raw_param(name: str, default: str) -> str:
    """Widget value, else ``MEDTECH_<NAME>`` env var, else ``default``."""
    if _DBUTILS is not None:
        try:
            return str(_DBUTILS.widgets.get(name))
        except Exception:  # widget not declared (e.g. job parameters) -> fall through
            pass
    return os.environ.get(_ENV_PREFIX + name.upper(), default)


def _as_bool(text: str, default: bool) -> bool:
    value = text.strip().lower()
    if value in {"1", "true", "yes", "y", "on"}:
        return True
    if value in {"0", "false", "no", "n", "off"}:
        return False
    return default


def _as_float(name: str, default: float, lo: float, hi: float, notes: list[str]) -> float:
    try:
        value = float(_raw_param(name, str(default)))
    except ValueError:
        notes.append(f"{name} is not a number; using {default}")
        return default
    clamped = max(lo, min(hi, value))
    if clamped != value:
        notes.append(f"{name} clamped to {clamped}")
    return clamped


def _project(name: str, default: str, notes: list[str]) -> str:
    value = _raw_param(name, default).strip() or default
    if not _PROJECT_RE.match(value):
        notes.append(f"{name} is not a valid Lakebase project id; using {default}")
        return default
    return value


def load_params() -> tuple[Params, list[str]]:
    """Read and validate every widget; invalid values fall back to safe defaults."""
    notes: list[str] = []
    try:
        lookback = int(float(_raw_param("lookback_hours", "24")))
    except ValueError:
        notes.append("lookback_hours is not a number; using 24")
        lookback = 24
    clamped = max(_LOOKBACK_MIN_H, min(_LOOKBACK_MAX_H, lookback))
    if clamped != lookback:
        notes.append(f"lookback_hours clamped to {clamped}")
    mode = _raw_param("mode", MODES[0]).strip().lower()
    if mode not in MODES:
        notes.append(f"mode={mode!r} is not one of {MODES}; using 'realtime'")
        mode = MODES[0]
    setup_id = _raw_param("setup_id", "").strip()
    if not _IDENT_RE.match(setup_id):
        notes.append("setup_id contains unsupported characters and was ignored")
        setup_id = ""
    # Outside Databricks no workspace is reachable, so default to the synthetic dataset.
    sample_default = "false" if IN_DATABRICKS else "true"
    params = Params(
        setup_id=setup_id,
        mode=mode,
        lookback_hours=clamped,
        redact=_as_bool(_raw_param("redact", "true"), True),
        sample=_as_bool(_raw_param("sample", sample_default), not IN_DATABRICKS),
        app_backend_project=_project("app_backend_project", DEFAULT_APP_BACKEND_PROJECT, notes),
        pipelines_project=_project("pipelines_project", DEFAULT_PIPELINES_PROJECT, notes),
        lot_cost=_as_float("lot_cost_usd", DEFAULT_LOT_COST_USD, 0.0, 10_000_000.0, notes),
        inspection_cost=_as_float(
            "inspection_cost_usd", DEFAULT_INSPECTION_COST_USD, 0.0, 10_000_000.0, notes
        ),
        capture_rate=_as_float("capture_rate", DEFAULT_CAPTURE_RATE, 0.0, 1.0, notes),
    )
    return params, notes


def missing_requirements() -> list[str]:
    """pip requirements the live (non-sample) mode needs but the compute lacks."""
    import importlib.metadata
    import importlib.util

    missing: list[str] = []
    try:
        parts = re.findall(r"\d+", importlib.metadata.version("databricks-sdk"))[:3]
        if tuple(int(p) for p in parts) < (0, 133, 0):
            missing.append("databricks-sdk>=0.133.0")
    except importlib.metadata.PackageNotFoundError:
        missing.append("databricks-sdk>=0.133.0")
    if importlib.util.find_spec("pg8000") is None:
        missing.append("pg8000")
    return missing


def ensure_dependencies(params: Params) -> None:
    """On Databricks (live mode only) install missing libraries once and restart Python."""
    if params.sample or not IN_DATABRICKS:
        return
    missing = missing_requirements()
    if not missing:
        return
    shell = globals().get("get_ipython", lambda: None)()
    if shell is None or _DBUTILS is None:
        print("Missing libraries (add them to the Environment): " + ", ".join(missing))
        return
    shell.run_line_magic("pip", "install -q " + " ".join(f"'{m}'" for m in missing))
    print("Installed " + ", ".join(missing) + "; restarting Python (re-run if it stops).")
    _DBUTILS.library.restartPython()


_declare_widgets()
PARAMS, PARAM_NOTES = load_params()
ensure_dependencies(PARAMS)

In [ ]:
from dataclasses import dataclass

# ── Access limits (the notebook is strictly read-only) ────────────────────────
STATEMENT_TIMEOUT_MS = 10_000  # Lakebase control-plane / metadata reads
HOT_STATEMENT_TIMEOUT_MS = 40_000  # bounded hot-tier reads (thousands of partitions to plan)
LAKEBASE_CONNECT_TIMEOUT_S = 10
LAKEBASE_ROW_LIMIT = 2_000
UC_ROW_LIMIT = 5_000
UC_WAIT_TIMEOUT = "50s"
HOT_LIVE_MINUTES = 15  # the hot tier is only asked for the newest minutes ("live now")

# ── Naming conventions mirrored from the app (src/iot_twin) ───────────────────
PG_DATABASE = "databricks_postgres"
APP_BACKEND_PROJECT_PREFIX = DEFAULT_APP_BACKEND_PROJECT  # <RESOURCE_PREFIX>-backend
APP_BACKEND_BRANCH = "production"
PRIMARY_ENDPOINT, REPLICA_ENDPOINT = "primary", "replica"
HOT_TABLE = "public.sensor_readings"
HISTORY_TABLE = f"lb__shadow_{INDUSTRY}_sensor_readings_history"  # realtime CDF mirror
METRICS_TABLE = f"{INDUSTRY}_metrics"
# Same row filter the app dashboard and Genie apply to the realtime CDF mirror
# (services/genie/genie_space_content.py telemetry_row_filter).
CDC_ROW_FILTER = f"_pg_change_type IN ('insert', 'update_postimage') AND industry = '{INDUSTRY}'"


# ── Plant model (mirror of src/iot_twin/domain/catalog.py, manufacturing) ─────
# Trinity Medical, Warsaw IN: MILL -> CMM INSPECT -> CLEAN / PASSIVATE -> STERILE PACK.
@dataclass(frozen=True)
class Asset:
    asset_id: str
    name: str
    role: str  # "cell" (production cell), "line", "lab" (CMM environment), "utility"
    component: str
    prefix: str  # sensor-name prefix that belongs to the asset


ASSETS: tuple[Asset, ...] = (
    Asset("mfg-asset-cnc-1", "CNC-1 5-axis mill", "cell", "Spindle 3", "cnc_1_"),
    Asset(
        "mfg-asset-cmm-1", "CMM-1 metrology lab", "lab", "Metrology lab environment", "hvac_zone_1_"
    ),
    Asset(
        "mfg-asset-press-1",
        "Clean & passivation line",
        "line",
        "Wash & passivation bath",
        "press_1_",
    ),
    Asset("mfg-asset-assy-1", "Sterile pack line", "line", "Pouch sealer drive", "assy_line_1_"),
    Asset(
        "mfg-asset-compressor-1", "Clean dry air compressor", "utility", "Air end", "compressor_1_"
    ),
)
FACILITY = "Facility (HVAC)"  # other hvac_* sensors: building services, not production equipment
# Not production equipment: excluded from the process-health index (section 1).
NON_PRODUCTION: tuple[str, ...] = (FACILITY, *(a.name for a in ASSETS if a.role == "lab"))
NON_PRODUCTION_SQL = ", ".join(f"'{name}'" for name in NON_PRODUCTION)
OTHER = "Other"


@dataclass(frozen=True)
class AlertRuleDef:
    """Alert rule as defined in the domain catalog (mirrored to intelligence_alert_rules)."""

    rule_id: str
    sensor_name: str
    operator: str  # "gt" or "lt"
    threshold: float
    severity: str
    fault_code: str


ALERT_RULES: tuple[AlertRuleDef, ...] = (
    AlertRuleDef(
        "mfg-alert-spindle-temp", "cnc_1_spindle_temp", "gt", 60.0, "crit", "VPP-SPINDLE3-TEMP"
    ),  # validated process parameter (OQ-TM-CNC1-014, PQ-TM-HS12-022)
    AlertRuleDef("mfg-alert-vibration", "cnc_1_vibration", "gt", 8.0, "warn", "SPC-SPINDLE3-VIBE"),
    AlertRuleDef(
        "mfg-alert-press", "press_1_pressure", "lt", 120.0, "warn", "CLN-PASSIVATION-PRESS-LOW"
    ),
)
RULE_BY_SENSOR = {r.sensor_name: r for r in ALERT_RULES}

# Key sensors for the health-trend section (KPI bindings + alert rules + spindle load).
KEY_SENSORS: tuple[tuple[str, str, str], ...] = (
    ("cnc_1_spindle_temp", "Spindle 3 temperature", "°C"),
    ("cnc_1_vibration", "Spindle 3 vibration", "mm/s"),
    ("cnc_1_spindle_power", "Spindle load (power)", "kW"),
    ("press_1_pressure", "Passivation line pressure", "bar"),
)

# Rate sensors used as throughput proxies: (asset, sensor, rate unit, per-hour multiplier, label).
THROUGHPUT_SENSORS: tuple[tuple[str, str, str, float, str], ...] = (
    ("Sterile pack line", "assy_line_1_speed", "m/min", 60.0, "conveyor metres per hour"),
    ("CNC-1 5-axis mill", "cnc_1_spindle_speed", "RPM", 60.0, "spindle revolutions per hour"),
)
COUNTER_SENSOR = "press_1_cycle_count"  # would give production counts if it incremented

# Drift flags (sensor-health section): last hour versus the whole window.
DRIFT_PCT_OF_NOMINAL = 5.0  # |mean(last hour) - mean(window)| / nominal, in %
BREACH_PCT_FLAG = 1.0  # share of readings past the alert threshold, in %

In [ ]:
import html as _html
import re
import traceback
from collections.abc import Iterable

_HOST_TLDS = r"(?:com|net|org|io|dev|ai|cloud)"
_REDACTION_RULES: tuple[tuple[str, re.Pattern[str], str], ...] = (
    # Secrets first: these are masked even when redaction is switched off.
    ("dsn", re.compile(r"\b[a-z][a-z0-9+.-]*://[^\s/:@]+:[^\s/@]+@[^\s'\"<>]+", re.I), "<dsn>"),
    ("jwt", re.compile(r"\beyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "<token>"),
    ("pat", re.compile(r"\b(?:dapi|dkea|dose|dsapi)[a-z0-9]{16,}\b", re.I), "<token>"),
    (
        "kv-secret",
        re.compile(
            r"(?i)\b(password|passwd|pwd|secret|client_secret|token|api[_-]?key|authorization)"
            r"(\s*[=:]\s*)(?:bearer\s+)?[^\s,;'\"<>)}\]]+"
        ),
        r"\1\2<redacted>",
    ),
    ("bearer", re.compile(r"(?i)\bbearer\s+[A-Za-z0-9._~+/=-]{12,}"), "Bearer <token>"),
)
_IDENTIFIER_RULES: tuple[tuple[str, re.Pattern[str], str], ...] = (
    (
        "arn",
        re.compile(r"\barn:aws[a-z-]*:[a-z0-9-]+:[a-z0-9-]*:\d{0,12}:[^\s'\"<>,;)]+", re.I),
        "<arn>",
    ),
    ("email", re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"), "<email>"),
    (
        "bucket-uri",
        re.compile(r"\b(s3a?|gs|abfss?|wasbs?|r2)://[^\s/'\"<>@]+(?:@[^\s/'\"<>]+)?", re.I),
        r"\1://<bucket>",
    ),
    (
        "workspace-path",
        re.compile(r"(?P<p>/(?:Workspace/)?(?:Users|Repos|Shared/\.[a-z]+))/[^/\s'\"<>]+"),
        r"\g<p>/<user>",
    ),
    ("url-host", re.compile(r"\b(https?)://[^\s/'\"<>]+", re.I), r"\1://<host>"),
    ("lakebase-ep", re.compile(r"(?<![A-Za-z0-9_-])ep-[a-z]+-[a-z]+-[a-z0-9]{4,}"), "<host>"),
    (
        "host",
        re.compile(
            rf"(?<![\w@<>.-])(?:[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?\.){{1,6}}{_HOST_TLDS}\b"
            r"(?::\d{2,5})?",
            re.I,
        ),
        "<host>",
    ),
    (
        "uuid",
        re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b", re.I),
        "<uuid>",
    ),
    ("hex-id", re.compile(r"\b[0-9a-f]{32}\b", re.I), "<hex-id>"),
    ("cluster-id", re.compile(r"\b\d{4}-\d{6}-[a-z0-9]{8}\b"), "<cluster-id>"),
    ("long-id", re.compile(r"(?<![\d,.])\d{10,}(?![\d,]|\.\d)"), "<id>"),
)
_MIN_KNOWN_LEN = 3
_TRACE_MAX_CHARS = 1200


class Redactor:
    """Central masking for everything the notebook renders.

    * Secret-looking values (tokens, passwords, DSNs) are *always* masked.
    * With ``enabled`` also hosts, ids, emails, ARNs/accounts, buckets and users are masked,
      plus every exact value registered through :meth:`register` (job ids, catalog names...).
    """

    def __init__(self, enabled: bool = True) -> None:
        self.enabled = enabled
        self._known: dict[str, str] = {}

    def register(self, value: object, label: str) -> None:
        """Mask a runtime-discovered exact value (and, for emails, its user-name part)."""
        text = str(value or "").strip()
        if len(text) < _MIN_KNOWN_LEN:
            return
        self._known[text] = label
        if "@" in text:
            local = text.split("@", 1)[0]
            if len(local) >= 4:
                self._known.setdefault(local, "<user>")

    def register_all(self, values: Iterable[object], label: str) -> None:
        for value in values:
            self.register(value, label)

    def _mask_known(self, text: str) -> str:
        # Longest first so "a.b.c" is masked before its parts.
        for value in sorted(self._known, key=len, reverse=True):
            pattern = rf"(?<![A-Za-z0-9_]){re.escape(value)}(?![A-Za-z0-9_])"
            text = re.sub(pattern, self._known[value], text, flags=re.I)
        return text

    def __call__(self, value: object) -> str:
        text = value if isinstance(value, str) else str(value)
        for _name, pattern, repl in _REDACTION_RULES:
            text = pattern.sub(repl, text)
        if not self.enabled:
            return text
        text = self._mask_known(text)
        for _name, pattern, repl in _IDENTIFIER_RULES:
            text = pattern.sub(repl, text)
        return text

    def exception(self, exc: BaseException, *, with_trace: bool = False) -> str:
        """Redacted, bounded rendering of an exception (including chained causes)."""
        if with_trace:
            raw = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
        else:
            raw = f"{type(exc).__name__}: {exc}"
            if exc.__cause__ is not None:
                raw += f" <- {type(exc.__cause__).__name__}: {exc.__cause__}"
        cleaned = self(raw)
        if len(cleaned) > _TRACE_MAX_CHARS:
            cleaned = cleaned[:_TRACE_MAX_CHARS] + " ...[truncated]"
        return cleaned

    def html(self, value: object) -> str:
        """Redact then HTML-escape plain text for embedding in markup."""
        return _html.escape(self(value))


REDACT = Redactor(enabled=True)

In [ ]:
import re

_ALLOWED_FIRST_KEYWORDS = frozenset({"select", "with", "describe", "show", "explain"})
_FORBIDDEN_KEYWORDS = re.compile(
    r"\b(insert|update|delete|merge|drop|alter|create|truncate|grant|revoke|copy|call|"
    r"optimize|vacuum|refresh|restore|analyze|reindex|execute|msck)\b",
    re.I,
)
_COMMENT_RE = re.compile(r"--[^\n]*|/\*.*?\*/", re.S)
_STRING_RE = re.compile(r"'(?:[^']|'')*'")
# Server-side functions that would write or change state even inside a SELECT.
_FORBIDDEN_FUNCTIONS = re.compile(
    r"\b(pg_terminate_backend|pg_cancel_backend|pg_reload_conf|pg_switch_wal|pg_create_\w+|"
    r"pg_drop_\w+|set_config|nextval|setval|lo_\w+|dblink\w*|"
    r"lakets\.(?:create|drop|add|remove|enable|disable|run|execute|refresh|reset)\w*)\s*\(",
    re.I,
)


class UnsafeSqlError(ValueError):
    """Raised when a statement is not strictly read-only."""


def assert_read_only(sql: str) -> str:
    """Return ``sql`` unchanged if it is a single read-only statement, else raise.

    Defence in depth on top of a read-only transaction / warehouse permissions:
    one statement, starts with SELECT/WITH/DESCRIBE/SHOW/EXPLAIN, no write/DDL keyword
    outside string literals and comments, no state-changing server functions.
    """
    stripped = _STRING_RE.sub("''", _COMMENT_RE.sub(" ", sql)).strip().rstrip(";").strip()
    if not stripped:
        raise UnsafeSqlError("empty statement")
    if ";" in stripped:
        raise UnsafeSqlError("multiple statements are not allowed")
    first = stripped.split(None, 1)[0].lower()
    if first not in _ALLOWED_FIRST_KEYWORDS:
        raise UnsafeSqlError(f"statement must start with SELECT/WITH/DESCRIBE/SHOW, got {first!r}")
    if match := _FORBIDDEN_KEYWORDS.search(stripped):
        raise UnsafeSqlError(f"forbidden keyword in read-only statement: {match.group(0)!r}")
    if match := _FORBIDDEN_FUNCTIONS.search(stripped):
        raise UnsafeSqlError(f"forbidden function in read-only statement: {match.group(1)!r}")
    return sql

In [ ]:
import math
import re
import sys
from datetime import UTC, datetime
from html import escape
from html.parser import HTMLParser
from typing import Any


def _display_html(markup: str) -> None:
    """Databricks: ``displayHTML``. Jupyter: a bundle with Markdown/plain-text fallbacks."""
    show = globals().get("displayHTML")
    if show is not None:
        show(escape_placeholders(markup))
        return
    from IPython.display import display

    display(text_bundle(markup), raw=True)


def text_bundle(markup: str) -> dict[str, str]:
    """``mime_bundle`` whose text versions are redacted again (defence in depth)."""
    bundle = mime_bundle(markup)
    for key in ("text/markdown", "text/plain"):
        bundle[key] = REDACT(bundle[key])
    return bundle


def emit_html(markup: str) -> None:
    """Render markup after central redaction (the only place HTML leaves the notebook)."""
    _display_html(REDACT(markup))


class _RedactingStream:
    """Wrap stdout/stderr so stray print()/warnings are redacted too."""

    def __init__(self, inner: Any) -> None:
        self._inner = inner

    def write(self, data: str) -> int:
        return self._inner.write(REDACT(data) if isinstance(data, str) else data)

    def __getattr__(self, name: str) -> Any:
        return getattr(self._inner, name)


def install_output_guards() -> None:
    """Redact raw stdout/stderr and uncaught-exception tracebacks (best effort)."""
    for name in ("stdout", "stderr"):
        stream = getattr(sys, name)
        if not isinstance(stream, _RedactingStream):
            setattr(sys, name, _RedactingStream(stream))
    ip = globals().get("get_ipython", lambda: None)()
    if ip is not None and hasattr(ip, "set_custom_exc"):

        def _handler(shell: Any, etype: Any, value: Any, tb: Any, tb_offset: Any = None) -> None:
            value.__traceback__ = tb
            print("Uncaught error (redacted):\n" + REDACT.exception(value, with_trace=True))

        ip.set_custom_exc((Exception,), _handler)


# ── Formatting ────────────────────────────────────────────────────────────────
def num(value: Any) -> float | None:
    """Coerce Decimal/str/int to float; None when impossible or NaN."""
    try:
        out = None if value is None else float(value)
    except (TypeError, ValueError):
        return None
    return None if out is None or math.isnan(out) else out


def fmt_num(value: Any, digits: int = 1) -> str:
    """Number with thousands separators; ``-`` for None/NaN."""
    if isinstance(value, bool):
        return str(value)
    number = num(value)
    if number is None:
        return "-" if value is None or not isinstance(value, str) else value
    if float(number).is_integer():
        return f"{int(number):,}"
    return f"{number:,.{digits}f}"


def fmt_pct(value: Any, digits: int = 1) -> str:
    """Fraction (0-1) as a percentage."""
    number = num(value)
    return "-" if number is None else f"{100 * number:,.{digits}f}%"


def fmt_seconds(value: Any) -> str:
    """Compact duration: 45s, 3.2 min, 5.1 h, 2.4 d."""
    secs = num(value)
    if secs is None:
        return "-"
    if abs(secs) < 90:
        return f"{secs:,.0f}s"
    if abs(secs) < 5400:
        return f"{secs / 60:,.1f} min"
    if abs(secs) < 129_600:
        return f"{secs / 3600:,.1f} h"
    return f"{secs / 86400:,.1f} d"


def fmt_ms(value: Any) -> str:
    ms = num(value)
    if ms is None:
        return "-"
    return f"{ms:,.0f} ms" if ms < 10_000 else f"{ms / 1000:,.1f} s"


def to_dt(value: Any) -> datetime | None:
    """Best-effort conversion of a datetime / ISO string / epoch to aware UTC."""
    if value is None:
        return None
    if isinstance(value, datetime):
        return value if value.tzinfo else value.replace(tzinfo=UTC)
    if isinstance(value, int | float):
        return datetime.fromtimestamp(float(value), UTC)
    text = str(value).strip().replace("Z", "+00:00").replace(" ", "T", 1)
    try:
        parsed = datetime.fromisoformat(text)
    except ValueError:
        return None
    return parsed if parsed.tzinfo else parsed.replace(tzinfo=UTC)


def fmt_ts(value: Any) -> str:
    stamp = to_dt(value)
    return "-" if stamp is None else stamp.astimezone(UTC).strftime("%Y-%m-%d %H:%M UTC")


def fmt_hour(value: Any) -> str:
    stamp = to_dt(value)
    return "-" if stamp is None else stamp.astimezone(UTC).strftime("%m-%d %H:00")


# ── HTML building blocks ──────────────────────────────────────────────────────
_INK = "color:inherit"
_CARD = "border:1px solid rgba(128,128,128,.35);border-radius:8px;padding:8px 12px;margin:6px 0"


class _Raw(str):
    """Marks a cell value as already-safe HTML."""


def raw(markup: str) -> _Raw:
    return _Raw(markup)


def fmt_cell(value: Any) -> str:
    if isinstance(value, datetime):
        return fmt_ts(value)
    return value if isinstance(value, str) else fmt_num(value)


def html_table(
    rows: list[dict[str, Any]], columns: list[str] | None = None, max_rows: int = 40
) -> str:
    """Escaped HTML table from dict rows."""
    if not rows:
        return '<p style="opacity:.7"><i>no rows</i></p>'
    cols = columns or list(rows[0])
    head = "".join(f'<th style="text-align:left;padding:3px 12px">{escape(c)}</th>' for c in cols)
    body = []
    for row in rows[:max_rows]:
        cells = "".join(
            '<td style="padding:3px 12px;font-variant-numeric:tabular-nums">'
            f"{row[c] if isinstance(row.get(c), _Raw) else escape(fmt_cell(row.get(c)))}</td>"
            for c in cols
        )
        body.append(f"<tr>{cells}</tr>")
    more = (
        f"<p><i>{len(rows) - max_rows} more rows not shown</i></p>" if len(rows) > max_rows else ""
    )
    return (
        '<table style="border-collapse:collapse;border-bottom:1px solid rgba(128,128,128,.4)">'
        f'<thead style="border-bottom:1px solid rgba(128,128,128,.6)"><tr>{head}</tr></thead>'
        f"<tbody>{''.join(body)}</tbody></table>{more}"
    )


def kv_table(pairs: list[tuple[str, Any]]) -> str:
    return html_table([{"item": k, "value": v} for k, v in pairs], ["item", "value"], 200)


def heading(text: str, level: int = 3) -> str:
    return f"<h{level} style='margin:12px 0 4px'>{escape(text)}</h{level}>"


def paragraph(text: str) -> str:
    return f"<p style='margin:4px 0;max-width:900px'>{escape(text)}</p>"


_NOTE_COLORS = {"INFO": "#2a78d6", "WARN": "#fab219", "NA": "#898781", "SAMPLE": "#ec835a"}


def note(text: str, kind: str = "INFO") -> str:
    color = _NOTE_COLORS.get(kind, _NOTE_COLORS["INFO"])
    return (
        f'<div style="border-left:4px solid {color};padding:3px 10px;margin:6px 0;max-width:900px">'
        f"{escape(text)}</div>"
    )


def sample_banner() -> str:
    if not PARAMS.sample:
        return ""
    return (
        '<div style="border:2px dashed #ec835a;padding:6px 10px;margin:4px 0;font-weight:700">'
        "SAMPLE DATA: synthetic dataset, no workspace was contacted. Numbers are illustrative."
        "</div>"
    )


def not_available(kpi: str, reason: str) -> str:
    """A clear 'not available from this pipeline' block instead of invented numbers."""
    return (
        f'<div style="{_CARD};border-style:dashed">'
        f"<b>{escape(kpi)}: not available from this pipeline</b>"
        f"<p style='margin:4px 0'>Because {escape(reason)}</p></div>"
    )


def data_source_note(tier: str, tables: list[str], window: str, extra: str = "") -> str:
    """The 'Data source' footer every KPI section ends with."""
    rows = [("tier", tier), ("table", ", ".join(tables) or "-"), ("time window", window)]
    if extra:
        rows.append(("note", extra))
    return heading("Data source", 4) + kv_table(rows)


def stat_tiles(tiles: list[tuple[str, str, str]]) -> str:
    """Row of KPI tiles: (label, value, caption)."""
    cells = "".join(
        f'<div style="{_CARD};min-width:150px;display:inline-block;vertical-align:top;'
        f'margin-right:8px"><div style="font-size:12px;opacity:.75">{escape(label)}</div>'
        f'<div style="font-size:22px;font-weight:600">{escape(value)}</div>'
        f'<div style="font-size:11px;opacity:.7">{escape(caption)}</div></div>'
        for label, value, caption in tiles
    )
    # A hidden table carries the same values for text-only viewers.
    table = html_table(
        [{"KPI": label, "value": value, "detail": caption} for label, value, caption in tiles],
        ["KPI", "value", "detail"],
    )
    return f"<div>{cells}</div><details><summary>table view</summary>{table}</details>"


# ── Text fallbacks (identical copy of ops/html_fallback.py; kept in sync by a test) ──
_BLOCK_TAGS = frozenset({"div", "p", "ul", "ol", "li", "section", "pre", "thead", "tbody"})
_SKIP_TAGS = frozenset({"script", "style"})
_SVG_NS = 'xmlns="http://www.w3.org/2000/svg"'
_PURE_SVG = re.compile(r"^\s*<svg\b[^>]*>.*</svg>\s*$", re.DOTALL)
_SPACES = re.compile(r"[ \t\r\f\v]+")
_BLANKS = re.compile(r"\n{3,}")
# Characters that would otherwise turn text such as ``<host>`` into (hidden) inline HTML.
_MD_SPECIAL = re.compile(r"[\\`*<>]")
# Element names the report can emit; any other bare ``<name>`` is a redaction placeholder.
_HTML_TAGS = frozenset(
    [
        "a",
        "abbr",
        "b",
        "blockquote",
        "br",
        "caption",
        "code",
        "col",
        "colgroup",
        "dd",
        "del",
        "desc",
        "details",
        "div",
        "dl",
        "dt",
        "em",
        "g",
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6",
        "hr",
        "i",
        "img",
        "li",
        "line",
        "ol",
        "p",
        "path",
        "polyline",
        "pre",
        "rect",
        "s",
        "small",
        "span",
        "strong",
        "sub",
        "summary",
        "sup",
        "svg",
        "table",
        "tbody",
        "td",
        "text",
        "tfoot",
        "th",
        "thead",
        "title",
        "tr",
        "tspan",
        "u",
        "ul",
    ]
)
_BARE_TAG = re.compile(r"<([a-z][a-z0-9-]*)>")


class _MarkdownBuilder(HTMLParser):
    """Collects text from the report's HTML subset and lays it out as Markdown."""

    def __init__(self, markdown: bool) -> None:
        super().__init__(convert_charrefs=True)
        self._markdown = markdown
        self.out: list[str] = []
        self._skip = 0
        self._heading = 0
        self._table: list[list[str]] | None = None
        self._cell: list[str] | None = None
        self._svg: dict[str, list[str]] | None = None
        self._svg_part = ""

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag in _SKIP_TAGS:
            self._skip += 1
        elif self._svg is not None:
            self._svg_part = tag if tag in ("title", "desc") else ""
        elif tag == "svg":
            self._svg = {"title": [], "desc": []}
        elif tag == "table":
            self._table = []
        elif tag == "tr" and self._table is not None:
            self._table.append([])
        elif tag in ("td", "th") and self._table is not None:
            self._cell = []
        elif re.fullmatch(r"h[1-6]", tag):
            self._heading = int(tag[1])
            self.out.append("\n\n" + "#" * self._heading + " ")
        elif tag == "br":
            self._write(" " if self._cell is not None else "\n")
        elif tag in _BLOCK_TAGS:
            self.out.append("\n")

    def handle_endtag(self, tag: str) -> None:
        if tag in _SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
        elif tag == "svg" and self._svg is not None:
            self._flush_svg()
        elif self._svg is not None:
            self._svg_part = ""
        elif tag in ("td", "th") and self._cell is not None and self._table:
            self._table[-1].append(" ".join("".join(self._cell).split()))
            self._cell = None
        elif tag == "table" and self._table is not None:
            self.out.append("\n\n" + _pipe_table(self._table) + "\n\n")
            self._table = None
        elif re.fullmatch(r"h[1-6]", tag) and self._heading:
            self._heading = 0
            self.out.append("\n\n")
        elif tag in _BLOCK_TAGS:
            self.out.append("\n")

    def handle_data(self, data: str) -> None:
        if self._skip:
            return
        if self._svg is not None:
            if self._svg_part:
                self._svg[self._svg_part].append(data)
            return
        self._write(_MD_SPECIAL.sub(r"\\\g<0>", data) if self._markdown else data)

    def _write(self, text: str) -> None:
        if self._cell is not None:
            self._cell.append(text)
        elif self._table is None:
            self.out.append(text)

    def _flush_svg(self) -> None:
        svg, self._svg = self._svg or {}, None
        title = " ".join("".join(svg.get("title", [])).split()) or "chart"
        desc = " ".join("".join(svg.get("desc", [])).split())
        self._write(f"\n[chart] {title}" + (f": {desc}" if desc else "") + "\n")


def escape_placeholders(markup: str) -> str:
    """Escape redaction placeholders (``<host>``) that would otherwise render as empty tags."""
    return _BARE_TAG.sub(
        lambda m: m.group(0) if m.group(1) in _HTML_TAGS else f"&lt;{m.group(1)}&gt;", markup
    )


def _pipe_table(rows: list[list[str]]) -> str:
    rows = [r for r in rows if r]
    if not rows:
        return ""
    width = max(len(r) for r in rows)
    padded = [[c.replace("|", "\\|") for c in r] + [""] * (width - len(r)) for r in rows]
    lines = ["| " + " | ".join(padded[0]) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in padded[1:]]
    return "\n".join(lines)


def _convert(markup: str, markdown: bool) -> str:
    builder = _MarkdownBuilder(markdown)
    builder.feed(markup)
    builder.close()
    lines = [_SPACES.sub(" ", line).strip() for line in "".join(builder.out).split("\n")]
    text = _BLANKS.sub("\n\n", "\n".join(lines)).strip()
    return text or "(empty output)"


def html_to_markdown(markup: str) -> str:
    """Markdown rendering of the report's HTML (text is escaped so nothing is hidden)."""
    return _convert(markup, markdown=True)


def html_to_text(markup: str) -> str:
    """Plain-text rendering: same layout as the Markdown, without escapes."""
    return _convert(markup, markdown=False)


def standalone_svg(markup: str) -> str | None:
    """The markup as a standalone SVG document when it is exactly one ``<svg>`` element."""
    if not _PURE_SVG.match(markup) or markup.count("<svg") != 1:
        return None
    svg = markup.strip()
    return svg if _SVG_NS in svg else svg.replace("<svg", f"<svg {_SVG_NS}", 1)


def mime_bundle(markup: str) -> dict[str, str]:
    """``text/html`` plus Markdown/plain-text fallbacks (and ``image/svg+xml`` for charts)."""
    markup = escape_placeholders(markup)
    bundle = {
        "text/html": markup,
        "text/markdown": html_to_markdown(markup),
        "text/plain": html_to_text(markup),
    }
    svg = standalone_svg(markup)
    if svg is not None:
        bundle["image/svg+xml"] = svg
    return bundle


# ── Dependency-free SVG charts (dataviz reference palette, light + dark) ──────
# Series slots in fixed order (blue, orange); critical red only marks thresholds. Ink and
# gridlines inherit the notebook's text colour (currentColor), so light and dark themes both work.
_CHART_STYLE = (
    '<style type="text/css">'
    ".kv-t{fill:currentColor}.kv-s{fill:currentColor;fill-opacity:.78}"
    ".kv-m{fill:currentColor;fill-opacity:.6}"
    ".kv-g{stroke:currentColor;stroke-opacity:.14}.kv-b{stroke:currentColor;stroke-opacity:.4}"
    ".kv-1{fill:#2a78d6;stroke:#2a78d6}.kv-2{fill:#eb6834;stroke:#eb6834}"
    ".kv-r{stroke:#d03b3b;fill:#d03b3b}.kv-ref{stroke:currentColor;fill:currentColor;"
    "stroke-opacity:.7;fill-opacity:.78}"
    "</style>"
)
_FONT = 'font-family="system-ui,-apple-system,Segoe UI,sans-serif"'


def _svg_open(title: str, desc: str, width: int, height: int) -> str:
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}" '
        f'width="{width}" height="{height}" viewBox="0 0 {width} {height}" '
        f'style="max-width:100%" {_FONT}>{_CHART_STYLE}'
        f"<title>{escape(title)}</title><desc>{escape(desc)}</desc>"
        f'<text x="0" y="16" font-size="14" font-weight="600" class="kv-t">{escape(title)}</text>'
    )


def _nice_max(value: float) -> float:
    """Round an axis maximum up to 1/2/2.5/5 x 10^k."""
    if value <= 0:
        return 1.0
    exp = 10 ** math.floor(math.log10(value))
    for step in (1, 2, 2.5, 5, 10):
        if value <= step * exp:
            return step * exp
    return 10 * exp


def _round_top_bar(x: float, y: float, w: float, h: float, r: float = 4.0) -> str:
    """Path of a vertical bar anchored to the baseline with a rounded data end (top)."""
    r = max(0.0, min(r, w / 2, h))
    return (
        f"M{x:.1f},{y + h:.1f}V{y + r:.1f}Q{x:.1f},{y:.1f} {x + r:.1f},{y:.1f}"
        f"H{x + w - r:.1f}Q{x + w:.1f},{y:.1f} {x + w:.1f},{y + r:.1f}V{y + h:.1f}Z"
    )


def _round_right_bar(x: float, y: float, w: float, h: float, r: float = 4.0) -> str:
    """Path of a horizontal bar anchored left with a rounded data end (right)."""
    r = max(0.0, min(r, h / 2, w))
    return (
        f"M{x:.1f},{y:.1f}H{x + w - r:.1f}Q{x + w:.1f},{y:.1f} {x + w:.1f},{y + r:.1f}"
        f"V{y + h - r:.1f}Q{x + w:.1f},{y + h:.1f} {x + w - r:.1f},{y + h:.1f}H{x:.1f}Z"
    )


def hbar_chart(
    rows: list[tuple[str, float, str]],
    *,
    title: str,
    x_max: float,
    desc: str,
    reference: tuple[float, str] | None = None,
    width: int = 640,
) -> str:
    """Horizontal bars (one series): (label, value, value text), value labels at bar ends."""
    if not rows:
        return note(f"{title}: no data in the selected window", "NA")
    left, right, top, row_h = 150, 70, 30, 30
    height = top + row_h * len(rows) + 22
    pw = width - left - right
    parts = [_svg_open(title, desc, width, height)]
    for i, (label, value, text) in enumerate(rows):
        y = top + i * row_h + 6
        w = pw * max(0.0, min(value, x_max)) / x_max
        parts.append(
            f'<text x="{left - 8}" y="{y + 13}" font-size="12" text-anchor="end" class="kv-s">'
            f"{escape(label)}</text>"
            f'<path d="{_round_right_bar(left, y, max(w, 1.0), 18)}" class="kv-1"/>'
            f'<text x="{left + w + 6:.1f}" y="{y + 13}" font-size="12" class="kv-t">'
            f"{escape(text)}</text>"
        )
    base_y = top + row_h * len(rows)
    parts.append(
        f'<line x1="{left}" x2="{left}" y1="{top}" y2="{base_y}" class="kv-b" stroke-width="1"/>'
    )
    if reference:
        x = left + pw * reference[0] / x_max
        parts.append(
            f'<line x1="{x:.1f}" x2="{x:.1f}" y1="{top - 4}" y2="{base_y}" class="kv-ref" '
            'stroke-width="1.5" stroke-dasharray="4 3"/>'
            f'<text x="{x:.1f}" y="{base_y + 14}" font-size="11" text-anchor="middle" '
            f'class="kv-s">{escape(reference[1])}</text>'
        )
    parts.append("</svg>")
    return "".join(parts)


def column_chart(
    bars: list[tuple[str, float]],
    *,
    title: str,
    y_label: str,
    desc: str,
    target: tuple[float, str] | None = None,
    width: int = 720,
    height: int = 242,
) -> str:
    """Vertical columns over time (one series) with an optional dashed target line."""
    if not bars:
        return note(f"{title}: no data in the selected window", "NA")
    left, right, top, bottom = 56, 16, 46, 30
    pw, ph = width - left - right, height - top - bottom
    y_hi = _nice_max(max([v for _, v in bars] + ([target[0] * 1.1] if target else [])))
    slot = pw / len(bars)
    gap = 2.0 if slot > 6 else 0.5
    parts = [_svg_open(title, desc, width, height)]
    for i in range(5):
        yv = y_hi * i / 4
        y = top + ph - ph * i / 4
        cls = "kv-b" if i == 0 else "kv-g"
        parts.append(
            f'<line x1="{left}" x2="{width - right}" y1="{y:.1f}" y2="{y:.1f}" class="{cls}" '
            'stroke-width="1"/>'
            f'<text x="{left - 6}" y="{y + 4:.1f}" font-size="10" text-anchor="end" '
            f'class="kv-m">{escape(fmt_num(yv, 0 if y_hi >= 10 else 1))}</text>'
        )
    for idx, (_label, value) in enumerate(bars):
        h = ph * max(value, 0.0) / y_hi
        x = left + idx * slot + gap / 2
        parts.append(
            f'<path d="{_round_top_bar(x, top + ph - h, max(slot - gap, 1.0), h)}" class="kv-1"/>'
        )
    if target:
        y = top + ph - ph * target[0] / y_hi
        parts.append(
            f'<line x1="{left}" x2="{width - right}" y1="{y:.1f}" y2="{y:.1f}" class="kv-ref" '
            'stroke-width="1.5" stroke-dasharray="5 4"/>'
            f'<text x="{width - right}" y="{y - 4:.1f}" font-size="11" text-anchor="end" '
            f'class="kv-s">{escape(target[1])}</text>'
        )
    step = max(1, len(bars) // 6)
    for idx in range(0, len(bars), step):
        x = left + idx * slot + slot / 2
        parts.append(
            f'<text x="{x:.1f}" y="{height - 12}" font-size="10" text-anchor="middle" '
            f'class="kv-m">{escape(bars[idx][0])}</text>'
        )
    parts.append(
        f'<text x="{left}" y="{top - 8}" font-size="11" class="kv-m">{escape(y_label)}</text></svg>'
    )
    return "".join(parts)


def line_chart(
    series: dict[str, list[tuple[float, float, str]]],
    *,
    title: str,
    y_label: str,
    desc: str,
    threshold: tuple[float, str] | None = None,
    width: int = 720,
    height: int = 232,
) -> str:
    """Up to two series over time ((x epoch, y, x label) points) with direct end labels."""
    pts = [p for line in series.values() for p in line]
    if not pts:
        return note(f"{title}: no data in the selected window", "NA")
    left, right, top, bottom = 56, 96, 46, 30
    pw, ph = width - left - right, height - top - bottom
    x_lo, x_hi = min(p[0] for p in pts), max(p[0] for p in pts)
    y_vals = [p[1] for p in pts] + ([threshold[0]] if threshold else [])
    y_lo_raw, y_hi_raw = min(y_vals), max(y_vals)
    pad = (y_hi_raw - y_lo_raw) * 0.1 or abs(y_hi_raw) * 0.1 or 1.0
    y_lo, y_hi = y_lo_raw - pad, y_hi_raw + pad

    def sx(x: float) -> float:
        return left + (pw * (x - x_lo) / (x_hi - x_lo) if x_hi > x_lo else pw / 2)

    def sy(y: float) -> float:
        return top + ph - ph * (y - y_lo) / (y_hi - y_lo)

    parts = [_svg_open(title, desc, width, height)]
    for i in range(5):
        yv = y_lo + (y_hi - y_lo) * i / 4
        cls = "kv-b" if i == 0 else "kv-g"
        parts.append(
            f'<line x1="{left}" x2="{left + pw}" y1="{sy(yv):.1f}" y2="{sy(yv):.1f}" '
            f'class="{cls}" stroke-width="1"/><text x="{left - 6}" y="{sy(yv) + 4:.1f}" '
            f'font-size="10" text-anchor="end" class="kv-m">{escape(fmt_num(yv, 1))}</text>'
        )
    if threshold:
        y = sy(threshold[0])
        parts.append(
            f'<line x1="{left}" x2="{left + pw}" y1="{y:.1f}" y2="{y:.1f}" class="kv-r" '
            'stroke-width="1.5" stroke-dasharray="5 4"/>'
            f'<text x="{left + pw + 4}" y="{y + 4:.1f}" font-size="10" class="kv-r" '
            f'stroke="none">{escape(threshold[1])}</text>'
        )
    legend_x = left
    for idx, (label, line) in enumerate(series.items()):
        cls = f"kv-{idx + 1}"
        ordered = sorted(line)
        if not ordered:
            continue
        coords = " ".join(f"{sx(x):.1f},{sy(y):.1f}" for x, y, _ in ordered)
        parts.append(
            f'<polyline fill="none" class="{cls}" stroke-width="2" stroke-linejoin="round" '
            f'points="{coords}" style="fill:none"/>'
        )
        last_x, last_y, _ = ordered[-1]
        parts.append(
            f'<circle cx="{sx(last_x):.1f}" cy="{sy(last_y):.1f}" r="4" class="{cls}" '
            'stroke="none"/>'
            f'<text x="{sx(last_x) + 8:.1f}" y="{sy(last_y) + 4 - idx * 2:.1f}" font-size="11" '
            f'class="kv-t">{escape(label)}</text>'
        )
        parts.append(
            f'<rect x="{legend_x}" y="{height - 12}" width="10" height="10" rx="2" class="{cls}"/>'
            f'<text x="{legend_x + 14}" y="{height - 3}" font-size="11" class="kv-s">'
            f"{escape(label)}</text>"
        )
        legend_x += 30 + 7 * len(label)
    x_first = sorted(pts)[0][2]
    x_last = sorted(pts)[-1][2]
    parts.append(
        f'<text x="{left}" y="{top + ph + 14}" font-size="10" class="kv-m">{escape(x_first)}</text>'
        f'<text x="{left + pw}" y="{top + ph + 14}" font-size="10" text-anchor="end" '
        f'class="kv-m">{escape(x_last)}</text>'
        f'<text x="{left}" y="{top - 8}" font-size="11" class="kv-m">{escape(y_label)}</text></svg>'
    )
    return "".join(parts)

In [ ]:
from collections.abc import Sequence
from datetime import timedelta

# ── Lot clock (mirror of src/iot_twin/domain/lot_context.py) ────────────────────────────
# No MES in the demo: the producer stamps a deterministic lot / part on every cnc_1_* reading.
LOT_CYCLE_S = 300  # one lot per Spindle 3 heat-event cycle
PART_CYCLE_S = 20  # one hip stem every 20 s
PARTS_PER_LOT = LOT_CYCLE_S // PART_CYCLE_S
LOT_PREFIX = "TM-HS12"
SPINDLE_RULE_ID = "mfg-alert-spindle-temp"  # validated process parameter, 60 °C
SPINDLE_SENSOR = "cnc_1_spindle_temp"
SPINDLE_LIMIT_C = 60.0
DETECT_MATCH_WINDOW_S = 180  # an alert opening is matched to the breach onset just before it
QUALITY_ACTIONS = ("quarantine_lot", "open_ncr", "request_cmm_inspection", "request_maintenance")


def safe_ratio(part: float | None, total: float | None) -> float | None:
    """part / total, or None when either side is missing or total is not positive."""
    if part is None or total is None or total <= 0:
        return None
    return part / total


def percentile(values: Iterable[Any], q: float) -> float | None:
    """Linear-interpolated percentile (q in 0..1) of the non-missing values."""
    data = sorted(v for v in (num(x) for x in values) if v is not None)
    if not data:
        return None
    pos = (len(data) - 1) * q
    lo = int(math.floor(pos))
    hi = min(lo + 1, len(data) - 1)
    return data[lo] + (data[hi] - data[lo]) * (pos - lo)


def window_bounds(hours: int, now: datetime | None = None) -> tuple[datetime, datetime]:
    end = now or datetime.now(UTC)
    return end - timedelta(hours=hours), end


DAY_S = 86_400
PROJECTED_RFT_LABEL = "Projected right-first-time"
PROJECTED_RFT_DEFINITION = (
    "Projected right-first-time = (parts started - parts at risk) / parts started, over the "
    "lots an excursion touched. Parts at risk are the parts whose cut overlapped the time "
    "Spindle 3 was outside its validated limit. Projected: no CMM results exist yet."
)  # verbatim from src/iot_twin/domain/lot_context.py


def lot_id_at(epoch_s: float) -> str:
    """Lot being machined at ``epoch_s``: ``<prefix>-<YYMMDD>-<cycle of the UTC day>``."""
    day = datetime.fromtimestamp(epoch_s, UTC).strftime("%y%m%d")
    return f"{LOT_PREFIX}-{day}-{int((epoch_s % DAY_S) // LOT_CYCLE_S):03d}"


def lot_start_at(epoch_s: float) -> float:
    return float(epoch_s - (epoch_s % LOT_CYCLE_S))


def parts_in_window(start: datetime, end: datetime) -> list[tuple[str, str]]:
    """(lot_id, part_serial) of every part whose cut overlaps [start, end] (lot_context rule)."""
    lo, hi = sorted((start.timestamp(), end.timestamp()))
    parts = []
    for slot in range(int(lo // PART_CYCLE_S), int(hi // PART_CYCLE_S) + 1):
        t0 = slot * PART_CYCLE_S
        lot = lot_id_at(t0)
        parts.append((lot, f"{lot}-{int((t0 % LOT_CYCLE_S) // PART_CYCLE_S) + 1:02d}"))
    return parts


def projected_rft(start: datetime, end: datetime, now: datetime) -> dict[str, int] | None:
    """Mirror of lot_context.projected_rft: parts started vs at risk over the touched lots."""
    at_risk = parts_in_window(start, end)
    if not at_risk:
        return None
    lo = min(start.timestamp(), end.timestamp())
    first_slot = int(lo // PART_CYCLE_S)
    cut_starts = [(first_slot + i) * PART_CYCLE_S for i in range(len(at_risk))]
    lot_starts = sorted({lot_start_at(t) for t in cut_starts})
    current = now.timestamp()
    started = sum(min(PARTS_PER_LOT, int((current - ls) // PART_CYCLE_S) + 1)
                  for ls in lot_starts if current >= ls)  # fmt: skip
    started = max(started, len(at_risk))
    return {"good": started - len(at_risk), "parts_started": started,
            "parts_at_risk": len(at_risk)}  # fmt: skip


def projected_rft_total(rows: Sequence[dict[str, int] | None]) -> dict[str, Any]:
    """Pooled projected RFT over every excursion: sum(good) / sum(parts started)."""
    good = sum(r["good"] for r in rows if r)
    started = sum(r["parts_started"] for r in rows if r)
    return {"good": good, "parts_started": started, "rft": safe_ratio(good, started)}


# ── Value model (executive view) ────────────────────────────────────────────────────────
def copq_avoided(
    events: int, capture_rate: float, lot_cost: float, inspection_cost: float
) -> float:
    """Cost of poor quality avoided = events x capture rate x (lot cost - inspection cost).

    ``events`` are detected excursions of the validated limit; ``capture_rate`` is the share of
    them that would otherwise have escaped to a scrapped (or shipped) lot; each captured event
    saves the lot but still pays for the inspection that dispositions it. Never negative.
    """
    return max(0.0, events * capture_rate * (lot_cost - inspection_cost))


def exposure_remaining(uncontained_lots: int, lot_cost: float) -> float:
    """Value of the lots touched by an excursion that have no quarantine decision yet."""
    return max(0, uncontained_lots) * lot_cost


# ── Alert episodes (operations view) ────────────────────────────────────────────────────
def build_episodes(events: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:
    """Pair each 'opened' event with the acknowledge / clear that follow it on the same alert.

    One alert row is reused for every episode of its rule, so an episode runs from an 'opened'
    event to the next 'opened' event of the same alert.
    """
    by_alert: dict[str, list[dict[str, Any]]] = {}
    for e in sorted(
        events, key=lambda r: to_dt(r["created_at"]) or datetime.min.replace(tzinfo=UTC)
    ):
        by_alert.setdefault(str(e["alert_id"]), []).append(e)
    episodes = []
    for alert_id, rows in by_alert.items():
        current: dict[str, Any] | None = None
        for e in rows:
            kind, at = e["event_type"], to_dt(e["created_at"])
            if kind == "opened":
                current = {"alert_id": alert_id, "rule_id": e.get("rule_id"),
                           "fault_code": e.get("fault_code"), "severity": e.get("severity"),
                           "opened_at": at, "acknowledged_at": None, "cleared_at": None}  # fmt: skip
                episodes.append(current)
            elif (
                current is not None
                and kind == "acknowledged"
                and current["acknowledged_at"] is None
            ):
                current["acknowledged_at"] = at
            elif current is not None and kind == "cleared" and current["cleared_at"] is None:
                current["cleared_at"] = at
    return sorted(episodes, key=lambda ep: ep["opened_at"])


def seconds_between(start: Any, end: Any) -> float | None:
    a, b = to_dt(start), to_dt(end)
    if a is None or b is None:
        return None
    return (b - a).total_seconds()


def match_onsets(episodes: Sequence[dict[str, Any]], onsets: Sequence[Any]) -> list[float | None]:
    """Time-to-detect per episode: opening minus the latest breach onset at or before it.

    An onset is the first second the raw telemetry crossed the limit; only onsets within
    ``DETECT_MATCH_WINDOW_S`` before the opening count (older ones belong to other episodes).
    """
    stamps = sorted(t for t in (to_dt(o) for o in onsets) if t is not None)
    out: list[float | None] = []
    for ep in episodes:
        opened = ep["opened_at"]
        best = None
        for t in stamps:
            if t <= opened + timedelta(seconds=1):  # 1 s slack for second-truncated onsets
                best = t
            else:
                break
        delta = (opened - best).total_seconds() if best else None
        out.append(
            max(0.0, delta) if delta is not None and delta <= DETECT_MATCH_WINDOW_S else None
        )
    return out


def latency_histogram(values: Iterable[Any], edges: Sequence[float]) -> list[tuple[str, int]]:
    """Counts per latency bucket (ms); the last bucket is open-ended."""
    data = [v for v in (num(x) for x in values) if v is not None]
    buckets = []
    for i, lo in enumerate(edges):
        hi = edges[i + 1] if i + 1 < len(edges) else None
        label = (
            f"{fmt_num(lo, 0)}-{fmt_num(hi, 0)} ms" if hi is not None else f">= {fmt_num(lo, 0)} ms"
        )
        count = sum(1 for v in data if v >= lo and (hi is None or v < hi))
        buckets.append((label, count))
    return buckets

In [ ]:
# Every statement is a SELECT; ``assert_read_only`` re-checks each one before it runs.
# Lakebase statements use ``%(name)s`` parameters (pg8000 pyformat); Unity Catalog statements are
# ``str.format`` templates whose identifiers are validated by ``uc_fqn`` before use.
_PG_WINDOW = "now() - make_interval(hours => %(hours)s)"

LAKEBASE_SQL: dict[str, tuple[str, str]] = {
    # name: (target, sql) -- app = control-plane Lakebase, hot = pipeline Lakebase (hot tier)
    "cp_setup_by_id": (
        "app",
        "SELECT id::text AS id, industry, status, warehouse_id, updated_at FROM pipeline_setup"
        " WHERE id::text = %(setup_id)s LIMIT 1",
    ),
    "cp_setup_latest": (
        "app",
        "SELECT id::text AS id, industry, status, warehouse_id, updated_at FROM pipeline_setup"
        " WHERE industry = %(industry)s ORDER BY updated_at DESC LIMIT 1",
    ),
    "cp_deployment": (
        "app",
        "SELECT id::text AS id, mode, status, cdf_enabled, lakebase_branch, delta_table,"
        " created_at, updated_at FROM pipeline_deployment"
        " WHERE setup_id::text = %(setup_id)s AND mode = %(mode)s"
        " ORDER BY updated_at DESC LIMIT 1",
    ),
    "cp_resources": (
        "app",
        "SELECT resource_type, identifier, parent_ref FROM provisioned_resource"
        " WHERE setup_id::text = %(setup_id)s AND resource_type IN"
        " ('uc_table', 'uc_schema', 'lakebase_project', 'lakebase_branch') LIMIT 500",
    ),
    # Every lifecycle event of every alert of the industry in the window (episodes are paired
    # in Python: an alert row is reused for each episode of its rule).
    "alert_events": (
        "app",
        "SELECT e.alert_id::text AS alert_id, e.event_type, e.created_at, a.rule_id,"
        " a.fault_code, a.severity FROM alert_event e JOIN alert a ON a.id = e.alert_id"
        f" WHERE a.industry = %(industry)s AND e.created_at >= {_PG_WINDOW}"
        " ORDER BY e.created_at LIMIT 2000",
    ),
    # Signed operator decisions (Part 11-style audit columns) with the lot facts they recorded.
    "responses": (
        "app",
        "SELECT r.alert_id::text AS alert_id, r.episode_opened_at, r.action, r.created_at,"
        " r.signature_meaning, r.signed_at, r.reference::text AS reference, a.rule_id"
        " FROM alert_response r JOIN alert a ON a.id = r.alert_id"
        f" WHERE a.industry = %(industry)s AND r.created_at >= {_PG_WINDOW}"
        " ORDER BY r.created_at LIMIT 2000",
    ),
    # Hot tier: only the newest row (cheap on thousands of partitions).
    "hot_latest": (
        "hot",
        "SELECT event_time AS max_t, extract(epoch FROM (now() - event_time)) AS lag_s"
        f" FROM {HOT_TABLE} ORDER BY event_time DESC LIMIT 1",
    ),
}

_UC_WINDOW = "event_time >= current_timestamp() - INTERVAL {hours} HOURS"

UC_SQL: dict[str, str] = {
    "glance": (
        "SELECT count(*) AS n, max(event_time) AS max_t, count(DISTINCT sensor_name) AS sensors,"
        " count_if(event_time >= current_timestamp() - INTERVAL 60 MINUTES) AS n_last_hour,"
        " unix_timestamp(current_timestamp()) - unix_timestamp(max(event_time)) AS lag_s"
        " FROM {table} WHERE {row_filter}" + _UC_WINDOW
    ),
    # Excursions of the validated limit, sessionised from the raw telemetry: consecutive breach
    # seconds less than 30 s apart form one excursion. onset = the first second over the limit.
    "excursions": (
        "WITH b AS (SELECT date_trunc('second', event_time) AS s, max(value) AS peak"
        " FROM {table} WHERE {row_filter}"
        + _UC_WINDOW
        + " AND sensor_name = '"
        + SPINDLE_SENSOR
        + "' AND value > "
        + str(SPINDLE_LIMIT_C)
        + " GROUP BY 1),"
        " g AS (SELECT s, peak, CASE WHEN lag(s) OVER (ORDER BY s) IS NULL OR"
        " unix_timestamp(s) - unix_timestamp(lag(s) OVER (ORDER BY s)) > 30 THEN 1 ELSE 0 END"
        " AS new_ex FROM b), e AS (SELECT s, peak, sum(new_ex) OVER (ORDER BY s) AS ex FROM g)"
        " SELECT ex, min(s) AS onset, max(s) AS last_breach, count(*) AS breach_seconds,"
        " max(peak) AS peak FROM e GROUP BY ex ORDER BY onset"
    ),
    # Lots and parts the mill produced in the window (lot context stamped on cnc_1_* readings).
    "production": (
        "SELECT count(DISTINCT get_json_object(metadata, '$.lot_id')) AS lots,"
        " count(DISTINCT get_json_object(metadata, '$.part_serial')) AS parts,"
        " min(event_time) AS first_t, max(event_time) AS last_t FROM {table}"
        " WHERE {row_filter}" + _UC_WINDOW + " AND startswith(sensor_name, 'cnc_1_')"
        " AND get_json_object(metadata, '$.lot_id') IS NOT NULL"
    ),
    # Per-batch latency of the stream in the window: percentiles across batches and the raw
    # values for the histogram (bounded by UC_ROW_LIMIT, newest first).
    "latency_summary": (
        "SELECT count(*) AS batches, min(timestamp) AS first_batch, max(timestamp) AS last_batch,"
        " percentile(e2e_p50, 0.5) AS p50_of_p50, percentile(e2e_p99, 0.5) AS p50_of_p99,"
        " percentile(e2e_p50, 0.99) AS p99_of_p50, percentile(e2e_p99, 0.99) AS p99_of_p99,"
        " max(e2e_p99) AS max_p99, sum(num_input_rows) AS rows_in,"
        " avg(processed_rps) AS processed_rps FROM {table}"
        " WHERE mode = '{mode}' AND e2e_p50 IS NOT NULL"
        " AND timestamp >= current_timestamp() - INTERVAL {hours} HOURS"
    ),
    "latency_batches": (
        "SELECT timestamp, e2e_p50, e2e_p99 FROM {table} WHERE mode = '{mode}'"
        " AND e2e_p50 IS NOT NULL AND timestamp >= current_timestamp() - INTERVAL {hours} HOURS"
        " ORDER BY timestamp DESC"
    ),
}

In [ ]:
from dataclasses import dataclass, field
from decimal import Decimal

PIPELINES_PROJECT_PREFIX = DEFAULT_PIPELINES_PROJECT  # <RESOURCE_PREFIX>-pipelines
_UC_IDENT_RE = re.compile(r"^[A-Za-z0-9_-]+$")


class Unavailable(Exception):
    """A data source could not be used; the message says why (shown to the reader)."""


def uc_fqn(*parts: str) -> str:
    """Validated, backtick-quoted three-part Unity Catalog name (hyphens are legal)."""
    if len(parts) != 3 or not all(_UC_IDENT_RE.match(p or "") for p in parts):
        raise ValueError("unsafe Unity Catalog identifier")
    return ".".join(f"`{p}`" for p in parts)


def classify_error(exc: BaseException) -> str:
    """Short, reader-friendly category for an access failure."""
    text = f"{type(exc).__name__} {exc}".lower()
    if isinstance(exc, Unavailable):
        return str(exc)
    if "timeout" in text or "canceling statement" in text:
        return "TIMEOUT"
    if "permission" in text or "denied" in text or "403" in text or "401" in text:
        return "NO ACCESS"
    if "not found" in text or "does not exist" in text or "404" in text:
        return "NOT FOUND"
    return "ERROR"


def to_plain(value: Any) -> Any:
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, bytes | memoryview):
        return bytes(value).decode("utf-8", "replace")
    return value


def plain_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [{k: to_plain(v) for k, v in row.items()} for row in rows]


@dataclass
class Pipeline:
    """What was resolved about the one medical-device pipeline in focus."""

    deployed: bool = False
    reason: str = ""
    status: str = "-"
    catalog: str = ""
    schema: str = ""
    cold_table: str = ""
    hot_branch: str = ""
    notes: list[str] = field(default_factory=list)

    @property
    def has_hot_tier(self) -> bool:
        return PARAMS.mode == "realtime"

    @property
    def has_metrics(self) -> bool:
        return PARAMS.mode in ("realtime", "microbatch")  # zerobus is not a Spark stream

    @property
    def row_filter(self) -> str:
        return f"{CDC_ROW_FILTER} AND " if PARAMS.mode == "realtime" else ""

    def fqn(self, table: str | None = None) -> str:
        return uc_fqn(self.catalog, self.schema, table or self.cold_table)

    def display(self, table: str | None = None) -> str:
        """Placeholder-safe table name for the data-source notes (redaction masks the parts)."""
        if not (self.catalog and self.schema):
            return "-"
        return f"{self.catalog}.{self.schema}.{table or self.cold_table}"


class LakebaseSession:
    """One read-only Postgres connection (OAuth token as password, TLS, short timeouts).

    Uses pg8000 (pure Python): loading psycopg2's native library can abort the Python process
    on serverless compute.
    """

    def __init__(self, host: str, user: str, token: str) -> None:
        import ssl

        import pg8000.dbapi

        pg8000.dbapi.paramstyle = "pyformat"  # the queries use %(name)s parameters
        self._conn = pg8000.dbapi.connect(
            user=user,
            host=host,
            port=5432,
            database=PG_DATABASE,
            password=token,
            ssl_context=ssl.create_default_context(),
            timeout=LAKEBASE_CONNECT_TIMEOUT_S + HOT_STATEMENT_TIMEOUT_MS // 1000,
            application_name="medtech-kpis",
            startup_params={
                "default_transaction_read_only": "on",
                "statement_timeout": str(STATEMENT_TIMEOUT_MS),
            },
        )

    def query(self, sql: str, params: dict[str, Any], timeout_ms: int) -> list[dict[str, Any]]:
        assert_read_only(sql)
        cur = self._conn.cursor()
        try:
            cur.execute("SET TRANSACTION READ ONLY")
            cur.execute(f"SET LOCAL statement_timeout = {int(timeout_ms)}")
            cur.execute(sql, params)
            names = [d[0] for d in cur.description or []]
            rows = cur.fetchmany(LAKEBASE_ROW_LIMIT)
            return plain_rows([dict(zip(names, row, strict=True)) for row in rows])
        finally:
            cur.close()
            self._conn.rollback()  # never leave a transaction open; nothing was written

    def close(self) -> None:
        try:
            self._conn.close()
        except Exception:
            pass


def _field(obj: Any, *path: str) -> Any:
    for attr in path:
        obj = getattr(obj, attr, None)
        if obj is None:
            return None
    return obj


class LiveBackend:
    """Read-only workspace access: SDK list/get calls, Lakebase SELECTs, UC SELECTs."""

    label = "live workspace"

    def __init__(self) -> None:
        from databricks.sdk import WorkspaceClient

        self.ws = WorkspaceClient()
        self.user = self.ws.current_user.me().user_name or ""
        REDACT.register(self.user, "<user>")
        REDACT.register(str(self.ws.config.host or ""), "<host>")
        self._targets: dict[str, tuple[str, str]] = {}
        self._sessions: dict[str, LakebaseSession] = {}
        self.warehouse_id = ""

    # ── Lakebase ──────────────────────────────────────────────────────────────
    def _projects(self, prefix: str) -> list[str]:
        ids = sorted(p.name.split("/", 1)[1] for p in self.ws.postgres.list_projects() if p.name)
        return [i for i in ids if i.startswith(prefix)]

    def _branch_names(self, project: str) -> set[str]:
        branches = self.ws.postgres.list_branches(parent=f"projects/{project}")
        return {b.name.rsplit("/", 1)[-1] for b in branches if b.name}

    def find_project(self, explicit: str, prefix: str, branch: str) -> str:
        if explicit:
            return explicit
        for project in self._projects(prefix):
            if branch in self._branch_names(project):
                return project
        raise Unavailable(f"NOT FOUND: no Lakebase project {prefix}* with branch {branch!r}")

    def locate(self, target: str, project: str, branch: str, endpoints: tuple[str, ...]) -> None:
        parent = f"projects/{project}/branches/{branch}"
        names = {
            e.name.rsplit("/", 1)[-1]: e.name
            for e in self.ws.postgres.list_endpoints(parent=parent)
        }
        REDACT.register(project, "<lakebase-project>")
        for endpoint_id in endpoints:
            if endpoint_id in names:
                host = _field(
                    self.ws.postgres.get_endpoint(name=names[endpoint_id]),
                    "status",
                    "hosts",
                    "host",
                )
                if host:
                    REDACT.register(str(host), "<host>")
                    self._targets[target] = (names[endpoint_id], str(host))
                    return
        raise Unavailable(f"NOT FOUND: no {endpoints} endpoint on branch {branch!r}")

    def _session(self, target: str) -> LakebaseSession:
        if target not in self._sessions:
            if target not in self._targets:
                raise Unavailable("NOT FOUND: Lakebase target was not resolved")
            endpoint, host = self._targets[target]
            token = self.ws.postgres.generate_database_credential(endpoint=endpoint).token
            self._sessions[target] = LakebaseSession(host, self.user, token)
        return self._sessions[target]

    def lb(self, name: str, **params: Any) -> list[dict[str, Any]]:
        target, sql = LAKEBASE_SQL[name]
        timeout = HOT_STATEMENT_TIMEOUT_MS if target == "hot" else STATEMENT_TIMEOUT_MS
        return self._session(target).query(sql, params, timeout)

    # ── Unity Catalog (notebook Spark first, else the setup's SQL warehouse) ──
    def uc(self, name: str, **fields: Any) -> list[dict[str, Any]]:
        sql = assert_read_only(UC_SQL[name].format(**fields))
        spark = globals().get("spark")
        if spark is not None:
            rows = spark.sql(sql).limit(UC_ROW_LIMIT).collect()
            return plain_rows([r.asDict(recursive=True) for r in rows])
        if not self.warehouse_id:
            raise Unavailable("NO ACCESS: no Spark session and no SQL warehouse recorded")
        return self._warehouse(sql)

    def _warehouse(self, sql: str) -> list[dict[str, Any]]:
        from databricks.sdk.service.sql import ExecuteStatementRequestOnWaitTimeout

        resp = self.ws.statement_execution.execute_statement(
            statement=sql,
            warehouse_id=self.warehouse_id,
            wait_timeout=UC_WAIT_TIMEOUT,
            on_wait_timeout=ExecuteStatementRequestOnWaitTimeout.CANCEL,
            row_limit=UC_ROW_LIMIT,
        )
        state = str(getattr(_field(resp, "status", "state"), "value", ""))
        if state != "SUCCEEDED":
            raise RuntimeError(f"statement {state}: {_field(resp, 'status', 'error', 'message')}")
        cols = _field(resp, "manifest", "schema", "columns") or []
        names = [c.name for c in cols]
        types = [str(getattr(c.type_name, "value", c.type_name)) for c in cols]
        out = []
        for row in _field(resp, "result", "data_array") or []:
            out.append({n: _coerce(v, t) for n, v, t in zip(names, row, types, strict=True)})
        return out

    # ── Pipeline resolution (control plane, read-only) ────────────────────────
    def resolve(self) -> Pipeline:
        pipe = Pipeline()
        project = self.find_project(
            PARAMS.app_backend_project, APP_BACKEND_PROJECT_PREFIX, APP_BACKEND_BRANCH
        )
        self.locate("app", project, APP_BACKEND_BRANCH, (PRIMARY_ENDPOINT,))
        if PARAMS.setup_id:
            setups = self.lb("cp_setup_by_id", setup_id=PARAMS.setup_id)
        else:
            setups = self.lb("cp_setup_latest", industry=INDUSTRY)
        if not setups:
            pipe.reason = "no medical-device pipeline setup exists in the app's control plane"
            return pipe
        setup = setups[0]
        REDACT.register(setup["id"], "<setup-id>")
        REDACT.register(setup.get("warehouse_id"), "<warehouse-id>")
        self.warehouse_id = str(setup.get("warehouse_id") or "")
        deployments = self.lb("cp_deployment", setup_id=setup["id"], mode=PARAMS.mode)
        resources = self.lb("cp_resources", setup_id=setup["id"])
        if not deployments:
            pipe.reason = f"the medical-device setup has no {PARAMS.mode} deployment"
            return pipe
        dep = deployments[0]
        REDACT.register(dep["id"], "<deployment-id>")
        pipe.status = str(dep.get("status") or "-")
        parts = str(dep.get("delta_table") or "").split(".")
        if len(parts) == 3 and all(parts):
            pipe.catalog, pipe.schema, pipe.cold_table = parts
        else:
            for row in resources:
                ident = str(row.get("identifier") or "").split(".")
                if row.get("resource_type") == "uc_schema" and len(ident) == 2:
                    pipe.catalog, pipe.schema = ident
        if PARAMS.mode == "realtime" or not pipe.cold_table:
            pipe.cold_table = (
                HISTORY_TABLE if PARAMS.mode == "realtime" else f"{INDUSTRY}_{PARAMS.mode}"
            )
        REDACT.register(pipe.catalog, "<catalog>")
        REDACT.register(pipe.schema, "<schema>")
        if not (pipe.catalog and pipe.schema):
            pipe.reason = "the deployment does not record its Unity Catalog location"
            return pipe
        pipe.deployed = pipe.status == "deployed"
        if not pipe.deployed:
            pipe.reason = f"the {PARAMS.mode} deployment status is '{pipe.status}', not 'deployed'"
        if pipe.has_hot_tier:
            pipe.hot_branch = str(dep.get("lakebase_branch") or INDUSTRY)
            ref = next(
                (str(r.get("parent_ref") or "") for r in resources
                 if r.get("resource_type") == "lakebase_branch"),
                "",
            )  # fmt: skip
            explicit = ref.split("/", 1)[1] if ref.startswith("projects/") else ""
            try:
                hot_project = self.find_project(
                    explicit or PARAMS.pipelines_project, PIPELINES_PROJECT_PREFIX, pipe.hot_branch
                )
                self.locate(
                    "hot", hot_project, pipe.hot_branch, (REPLICA_ENDPOINT, PRIMARY_ENDPOINT)
                )
            except Exception as exc:
                pipe.notes.append(f"hot tier not reachable: {classify_error(exc)}")
        return pipe

    def close(self) -> None:
        for session in self._sessions.values():
            session.close()


def _coerce(value: Any, type_name: str) -> Any:
    """Statement Execution returns strings; restore numbers and timestamps."""
    if value is None:
        return None
    if type_name in {"INT", "LONG", "SHORT", "BYTE"}:
        return int(value)
    if type_name in {"DOUBLE", "FLOAT", "DECIMAL"}:
        return float(value)
    if type_name == "TIMESTAMP":
        return to_dt(value)
    if type_name == "BOOLEAN":
        return str(value).lower() == "true"
    return value

In [ ]:
import json
import random

_SAMPLE_CYCLE_S = LOT_CYCLE_S  # the producer's Spindle 3 heat-event cycle
_SAMPLE_BREACH_S = 70  # seconds over the limit per excursion
_SAMPLE_READINGS_PER_S = 40  # telemetry rows per second (all sensors)


class SampleBackend:
    """Deterministic synthetic implant cell: answers the same queries as the live backend."""

    label = "synthetic sample"
    user = ""

    def __init__(self) -> None:
        self.now = datetime.now(UTC).replace(microsecond=0)
        self.hours = PARAMS.lookback_hours
        self._rng = random.Random(20261120)
        self.warehouse_id = ""
        self._events, self._responses, self._onsets = self._generate()

    def _generate(self) -> tuple[list[dict[str, Any]], list[dict[str, Any]], list[dict[str, Any]]]:
        start = self.now - timedelta(hours=self.hours)
        first = (int(start.timestamp()) // _SAMPLE_CYCLE_S + 1) * _SAMPLE_CYCLE_S
        events, responses, onsets = [], [], []
        alert_id = "00000000-0000-4000-8000-000000000001"
        for i, t0 in enumerate(range(first, int(self.now.timestamp()) - 240, _SAMPLE_CYCLE_S)):
            onset = datetime.fromtimestamp(t0 + 120, UTC)
            opened = onset + timedelta(seconds=self._rng.uniform(1.0, 4.0))
            onsets.append({"ex": i, "onset": onset, "breach_seconds": _SAMPLE_BREACH_S,
                           "last_breach": onset + timedelta(seconds=_SAMPLE_BREACH_S),
                           "peak": 66 + self._rng.uniform(0, 6)})  # fmt: skip
            base = {"alert_id": alert_id, "rule_id": SPINDLE_RULE_ID,
                    "fault_code": "VPP-SPINDLE3-TEMP", "severity": "crit"}  # fmt: skip
            events.append({**base, "event_type": "opened", "created_at": opened})
            acked = i % 4 != 3  # one episode in four is never acknowledged
            if acked:
                ack = opened + timedelta(seconds=self._rng.uniform(25, 140))
                events.append({**base, "event_type": "acknowledged", "created_at": ack})
            cleared = onset + timedelta(seconds=_SAMPLE_BREACH_S + self._rng.uniform(5, 20))
            events.append({**base, "event_type": "cleared", "created_at": cleared})
            if acked and i % 2 == 0:  # half of the acknowledged episodes get a QA decision
                parts = parts_in_window(opened, cleared)
                lots = sorted({lot for lot, _ in parts})
                for k, action in enumerate(
                    ("quarantine_lot", "request_cmm_inspection", "open_ncr")
                ):
                    if action == "open_ncr" and i % 3:
                        continue
                    ref = {"lot_ids": lots, "parts_at_risk": [p for _, p in parts],
                           "parts_released": len(lots) * PARTS_PER_LOT - len(parts)}  # fmt: skip
                    responses.append({"alert_id": alert_id, "episode_opened_at": opened,
                                      "action": action, "rule_id": SPINDLE_RULE_ID,
                                      "created_at": ack + timedelta(seconds=10 + k),
                                      "signature_meaning": "approved", "signed_at": ack,
                                      "reference": json.dumps(ref)})  # fmt: skip
        return events, responses, onsets

    def resolve(self) -> Pipeline:
        REDACT.register("sample_catalog", "<catalog>")
        REDACT.register("sample_schema", "<schema>")
        return Pipeline(
            deployed=True, status="deployed", catalog="sample_catalog", schema="sample_schema",
            cold_table=HISTORY_TABLE if PARAMS.mode == "realtime" else f"{INDUSTRY}_{PARAMS.mode}",
            hot_branch=INDUSTRY,
        )  # fmt: skip

    def close(self) -> None:
        return None

    # ── Lakebase (control plane + hot tier) ─────────────────────────────────────────────
    def lb(self, name: str, **params: Any) -> list[dict[str, Any]]:
        if name == "alert_events":
            return list(self._events)
        if name == "responses":
            return list(self._responses)
        if name == "hot_latest":
            return [{"max_t": self.now - timedelta(seconds=2), "lag_s": 2.0}]
        raise Unavailable(f"NOT FOUND: sample backend has no Lakebase query {name!r}")

    # ── Unity Catalog ────────────────────────────────────────────────────────────────────
    def uc(self, name: str, **fields: Any) -> list[dict[str, Any]]:
        seconds = self.hours * 3600
        if name == "glance":
            return [{"n": seconds * _SAMPLE_READINGS_PER_S, "max_t": self.now, "sensors": 19,
                     "n_last_hour": 3600 * _SAMPLE_READINGS_PER_S, "lag_s": 3.0}]  # fmt: skip
        if name == "excursions":
            return list(self._onsets)
        if name == "production":
            return [{"lots": seconds // LOT_CYCLE_S, "parts": seconds // PART_CYCLE_S,
                     "first_t": self.now - timedelta(hours=self.hours), "last_t": self.now}]  # fmt: skip
        batches = self._batches()
        if name == "latency_batches":
            return batches
        if name == "latency_summary":
            p50s, p99s = [b["e2e_p50"] for b in batches], [b["e2e_p99"] for b in batches]
            return [{"batches": len(batches), "first_batch": batches[-1]["timestamp"],
                     "last_batch": batches[0]["timestamp"],
                     "p50_of_p50": percentile(p50s, 0.5), "p50_of_p99": percentile(p99s, 0.5),
                     "p99_of_p50": percentile(p50s, 0.99), "p99_of_p99": percentile(p99s, 0.99),
                     "max_p99": max(p99s), "rows_in": len(batches) * 9_000,
                     "processed_rps": 1_900.0}]  # fmt: skip
        raise Unavailable(f"NOT FOUND: sample backend has no Unity Catalog query {name!r}")

    def _batches(self) -> list[dict[str, Any]]:
        rng = random.Random(7)
        out = []
        for k in range(min(self.hours * 720, UC_ROW_LIMIT)):  # one batch every 5 s
            p50 = max(5.0, rng.lognormvariate(math.log(40), 0.35))
            tail = rng.lognormvariate(math.log(4), 0.6) if k % 97 else 25.0  # rare long tail
            out.append({"timestamp": self.now - timedelta(seconds=5 * k), "e2e_p50": p50,
                        "e2e_p99": p50 * (1.5 + tail)})  # fmt: skip
        return out

## Which pipeline is this?

The notebook reads the medical-device (`manufacturing`) pipeline from the app's own records (or `setup_id`), then only what it wrote in the window.

In [ ]:
REDACT.enabled = PARAMS.redact
install_output_guards()
RUN_STARTED = datetime.now(UTC)
WINDOW_START, WINDOW_END = window_bounds(PARAMS.lookback_hours, RUN_STARTED)
WINDOW_TEXT = f"last {PARAMS.lookback_hours} h ({fmt_ts(WINDOW_START)} to {fmt_ts(WINDOW_END)})"


class Data:
    """Cached, source-tagged access to the backend (each query runs at most once)."""

    def __init__(self, backend: Any, pipe: Pipeline) -> None:
        self.backend, self.pipe = backend, pipe
        self._cache: dict[tuple, tuple[str, Any]] = {}
        self.sources: dict[str, str] = {}

    def _run(self, key: tuple, source: str, fn: Any) -> list[dict[str, Any]]:
        if key not in self._cache:
            try:
                if self.backend is None:
                    raise Unavailable(f"NO ACCESS: {self.pipe.reason}")
                self._cache[key] = ("ok", fn())
                self.sources[source] = "reached"
            except Exception as exc:
                self._cache[key] = ("err", exc)
                if self.sources.get(source) != "reached":
                    self.sources[source] = classify_error(exc)
        kind, payload = self._cache[key]
        if kind == "err":
            raise payload
        return payload

    def cold(self, name: str) -> list[dict[str, Any]]:
        """UC Delta cold-tier query over the pipeline's telemetry table and lookback window."""
        if not self.pipe.catalog:
            raise Unavailable(f"NOT DEPLOYED: {self.pipe.reason}")
        fields = {"table": self.pipe.fqn(), "row_filter": self.pipe.row_filter,
                  "hours": PARAMS.lookback_hours}  # fmt: skip
        return self._run(
            ("uc", name), "UC Delta (cold tier)", lambda: self.backend.uc(name, **fields)
        )

    def latency(self, name: str) -> list[dict[str, Any]]:
        """Per-batch latency rows from the streaming metrics table."""
        if not self.pipe.has_metrics:
            raise Unavailable("NOT APPLICABLE: zerobus writes straight to Delta (no Spark stream)")
        if not self.pipe.catalog:
            raise Unavailable(f"NOT DEPLOYED: {self.pipe.reason}")
        fields = {"table": self.pipe.fqn(METRICS_TABLE), "mode": PARAMS.mode,
                  "hours": PARAMS.lookback_hours}  # fmt: skip
        return self._run(("metrics", name), "streaming metrics table",
                         lambda: self.backend.uc(name, **fields))  # fmt: skip

    def control(self, name: str) -> list[dict[str, Any]]:
        params = {"industry": INDUSTRY, "hours": PARAMS.lookback_hours}
        return self._run(("cp", name), "control-plane Lakebase (alerts, decisions)",
                         lambda: self.backend.lb(name, **params))  # fmt: skip

    def hot(self, name: str) -> list[dict[str, Any]]:
        if not self.pipe.has_hot_tier:
            raise Unavailable(f"NOT APPLICABLE: {PARAMS.mode} has no Lakebase hot tier")
        params = {"hours": PARAMS.lookback_hours}
        return self._run(("hot", name), "Lakebase LakeTS (hot tier)",
                         lambda: self.backend.lb(name, **params))  # fmt: skip


def connect() -> tuple[Any, Pipeline]:
    try:
        backend = SampleBackend() if PARAMS.sample else LiveBackend()
    except Exception as exc:
        return None, Pipeline(reason=f"the workspace is not reachable ({classify_error(exc)})")
    try:
        return backend, backend.resolve()
    except Exception as exc:
        reason = f"the app's control plane could not be read ({classify_error(exc)})"
        return backend, Pipeline(reason=reason)


def failure_text(exc: BaseException) -> str:
    """Why a data source could not answer (redacted, one line)."""
    category = classify_error(exc)
    if isinstance(exc, Unavailable):
        return category
    return f"{category}: {REDACT.exception(exc)[:240]}"


def not_deployed_block(section: str) -> str:
    return not_available(
        section,
        f"{PIPE.reason}. Deploy the medical-device ({INDUSTRY}) {PARAMS.mode} pipeline from the "
        "app's admin page, or pick another mode with the 'mode' widget.",
    )


def render_header() -> str:
    mode_label = {"realtime": "real-time (Spark Real-Time Mode)", "microbatch": "micro-batch",
                  "zerobus": "Zerobus direct ingest"}[PARAMS.mode]  # fmt: skip
    rows = [
        ("pipeline", f"Trinity Medical implant cell ({INDUSTRY}) / {mode_label}"),
        ("deployment status", PIPE.status),
        ("telemetry table (cold tier)", PIPE.display() if PIPE.catalog else "-"),
        ("time window", WINDOW_TEXT),
        (
            "value model",
            f"lot cost ${fmt_num(PARAMS.lot_cost, 0)}, inspection "
            f"${fmt_num(PARAMS.inspection_cost, 0)}, capture rate "
            f"{fmt_pct(PARAMS.capture_rate, 0)} (widgets)",
        ),  # fmt: skip
        ("data", BACKEND.label if BACKEND else "no workspace access"),
        ("redaction", "on" if REDACT.enabled else "off"),
    ]
    blocks = [sample_banner(), heading("Run summary", 3), kv_table(rows)]
    blocks += [note(n, "WARN") for n in PARAM_NOTES + PIPE.notes]
    if not PIPE.deployed:
        blocks.append(not_deployed_block(f"Medical-device {PARAMS.mode} pipeline"))
    return "".join(blocks)


BACKEND, PIPE = connect()
DATA = Data(BACKEND, PIPE)
SUMMARY: list[dict[str, Any]] = []  # headline numbers, collected by every section


def _episodes() -> list[dict[str, Any]]:
    return build_episodes(DATA.control("alert_events"))


def _spindle(episodes: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [ep for ep in episodes if ep.get("rule_id") == SPINDLE_RULE_ID]


def _responses() -> list[dict[str, Any]]:
    out = []
    for r in DATA.control("responses"):
        ref = r.get("reference")
        try:
            ref = json.loads(ref) if isinstance(ref, str) else dict(ref or {})
        except ValueError:
            ref = {}
        out.append({**r, "reference": ref})
    return out


emit_html(render_header())

item,value
pipeline,Trinity Medical implant cell (manufacturing) / real-time (Spark Real-Time Mode)
deployment status,deployed
telemetry table (cold tier),<catalog>.<schema>.lb__shadow_manufacturing_sensor_readings_history
time window,last 2 h (2026-11-20 14:21 UTC to 2026-11-20 16:21 UTC)
value model,"lot cost $24,000, inspection $1,500, capture rate 80% (widgets)"
data,live workspace
redaction,on


## Pipeline at a glance

Before trusting any KPI: how fresh the data is and how much of it there is.

In [ ]:
def pipeline_at_a_glance() -> None:
    """Freshness and volume: is the data current enough to trust the KPIs below?"""
    rows: list[tuple[str, str, str]] = []
    notes: list[str] = []
    try:
        g = DATA.cold("glance")[0]
        n, recent = num(g.get("n")) or 0, num(g.get("n_last_hour")) or 0
        lag = num(g.get("lag_s"))
        rows.append(("Data freshness (Delta)", fmt_seconds(max(lag, 0) if lag is not None else None),
                     "age of the newest reading in the cold tier"))  # fmt: skip
        rows.append(("Ingest rate", f"{fmt_num(recent / 60, 0)} rows/min", "last 60 min"))
        rows.append(("Readings in window", fmt_num(n), f"{fmt_num(g.get('sensors'))} sensors"))
        SUMMARY.append({"KPI": "Readings analysed", "value": fmt_num(n), "section": "glance"})
    except Exception as exc:
        notes.append(f"Delta cold tier: {failure_text(exc)}")
    try:
        latest = DATA.hot("hot_latest")
        if latest:
            lag = num(latest[0].get("lag_s"))
            rows.insert(0, ("Data freshness (Lakebase)",
                            fmt_seconds(max(lag, 0) if lag is not None else None),
                            "age of the newest hot-tier reading"))  # fmt: skip
    except Exception as exc:
        notes.append(f"Lakebase hot tier: {failure_text(exc)}")
    blocks = [heading("Pipeline at a glance", 2)]
    if not rows and not PIPE.deployed:
        blocks.append(not_deployed_block("Pipeline at a glance"))
    if rows:
        blocks.append(stat_tiles(rows))
    blocks += [note(n, "NA") for n in notes]
    tables = [PIPE.display()] if PIPE.catalog else []
    blocks.append(data_source_note("UC Delta cold tier + Lakebase hot tier", tables, WINDOW_TEXT))
    emit_html("".join(blocks))


pipeline_at_a_glance()

## Pipeline at a glance

Data freshness (Lakebase)

0s

age of the newest hot-tier reading

Data freshness (Delta)

3s

age of the newest reading in the cold tier

Ingest rate

16,813 rows/min

last 60 min

Readings in window

2,607,155

23 sensors

table view

| KPI | value | detail |
|---|---|---|
| Data freshness (Lakebase) | 0s | age of the newest hot-tier reading |
| Data freshness (Delta) | 3s | age of the newest reading in the cold tier |
| Ingest rate | 16,813 rows/min | last 60 min |
| Readings in window | 2,607,155 | 23 sensors |

#### Data source

| item | value |
|---|---|
| tier | UC Delta cold tier + Lakebase hot tier |
| table | <catalog>.<schema>.lb__shadow_manufacturing_sensor_readings_history |
| time window | last 2 h (2026-11-20 14:21 UTC to 2026-11-20 16:21 UTC) |

## EXECUTIVE (Alex Morgan, VP Quality & Site Head)

Cost of poor quality avoided, lots quarantined versus released and the exposure still open. The value model is a parameterised estimate: change `lot_cost_usd`, `inspection_cost_usd` and `capture_rate` to the site's numbers; the sensitivity table shows how much the capture-rate assumption matters.

In [ ]:
def _quarantine_facts(responses: list[dict[str, Any]]) -> dict[str, Any]:
    """Lots / parts contained by signed quarantine decisions (facts the app recorded)."""
    lots: set[str] = set()
    parts_held, parts_released, episodes = 0, 0, set()
    for r in responses:
        if r.get("action") != "quarantine_lot":
            continue
        ref = r["reference"]
        lots.update(str(x) for x in ref.get("lot_ids") or [])
        parts_held += len(ref.get("parts_at_risk") or [])
        parts_released += int(num(ref.get("parts_released")) or 0)
        episodes.add((r.get("alert_id"), str(to_dt(r.get("episode_opened_at")))))
    return {"lots": lots, "parts_held": parts_held, "parts_released": parts_released,
            "episodes": episodes}  # fmt: skip


def _uncontained_lots(episodes: list[dict[str, Any]], contained: set) -> set[str]:
    """Lots touched by a Spindle 3 excursion whose episode has no quarantine decision."""
    lots: set[str] = set()
    for ep in episodes:
        if (ep["alert_id"], str(ep["opened_at"])) in contained:
            continue
        end = ep.get("cleared_at") or RUN_STARTED
        lots.update(lot for lot, _ in parts_in_window(ep["opened_at"], end))
    return lots


def executive_section() -> None:
    blocks = [heading("EXECUTIVE: cost of poor quality and lot exposure", 2),
              paragraph("For <b>Alex Morgan, VP Quality &amp; Site Head</b>. Every Spindle 3 "
                        "excursion of the 60 °C validated limit puts the parts cut during it at "
                        "risk. Containing them early (quarantine, CMM, disposition) costs an "
                        "inspection; missing them costs the lot.")]  # fmt: skip
    try:
        episodes = _spindle(_episodes())
        responses = _responses()
    except Exception as exc:
        blocks.append(not_available("Executive KPIs", failure_text(exc)))
        emit_html("".join(blocks))
        return
    events = len(episodes)
    facts = _quarantine_facts(responses)
    contained = facts["episodes"]
    uncontained = _uncontained_lots(episodes, contained)
    try:
        prod = DATA.cold("production")[0]
        lots_made = int(num(prod.get("lots")) or 0)
    except Exception as exc:
        prod, lots_made = {}, None
        blocks.append(note(f"Lot volume from telemetry: {failure_text(exc)}", "NA"))
    avoided = copq_avoided(events, PARAMS.capture_rate, PARAMS.lot_cost, PARAMS.inspection_cost)
    exposure = exposure_remaining(len(uncontained), PARAMS.lot_cost)
    released = None if lots_made is None else max(0, lots_made - len(facts["lots"]))
    realised = safe_ratio(len(contained), events)
    blocks.append(stat_tiles([
        ("Cost of poor quality avoided", f"${fmt_num(avoided, 0)}",
         f"{events} excursions x {fmt_pct(PARAMS.capture_rate, 0)} capture x "
         f"(${fmt_num(PARAMS.lot_cost, 0)} - ${fmt_num(PARAMS.inspection_cost, 0)})"),
        ("Lots quarantined", fmt_num(len(facts["lots"]), 0),
         f"{fmt_num(facts['parts_held'], 0)} parts held, {fmt_num(facts['parts_released'], 0)} "
         "released from the same lots"),
        ("Lots released", fmt_num(released, 0) if released is not None else "-",
         f"of {fmt_num(lots_made, 0) if lots_made is not None else '-'} lots machined in the window"),
        ("Exposure remaining", f"${fmt_num(exposure, 0)}",
         f"{len(uncontained)} excursion lots without a quarantine decision"),
    ]))  # fmt: skip
    blocks.append(paragraph(
        f"<b>Containment realised:</b> {len(contained)} of {events} excursions "
        f"({fmt_pct(realised, 0) if realised is not None else '-'}) carry a signed quarantine "
        "decision. The value model applies the capture-rate assumption to <i>every</i> detected "
        "excursion (what detection makes possible); exposure remaining is the value still at "
        "risk until each excursion lot is dispositioned."))  # fmt: skip
    sens = [{"capture rate": fmt_pct(c, 0),
             "COPQ avoided": f"${fmt_num(copq_avoided(events, c, PARAMS.lot_cost, PARAMS.inspection_cost), 0)}"}
            for c in (0.5, 0.7, PARAMS.capture_rate, 0.9, 1.0)]  # fmt: skip
    blocks.append(heading("Sensitivity of the value model", 4) + html_table(sens))
    blocks.append(kv_table([
        ("formula", "events x capture rate x (lot cost - inspection cost)"),
        ("events", f"{events} Spindle 3 excursion episodes (alert rule {SPINDLE_RULE_ID}) opened"),
        ("lot cost / inspection cost", "illustrative defaults; set the widgets to site values"),
        ("lots quarantined", "distinct lot_ids recorded by signed quarantine_lot decisions"),
        ("lots released", "lots machined (distinct lot_id on cnc_1_* readings) minus quarantined"),
        ("exposure remaining", "lots cut during excursions with no quarantine decision x lot cost"),
    ]))  # fmt: skip
    SUMMARY.extend([
        {"KPI": "COPQ avoided (model)", "value": f"${fmt_num(avoided, 0)}", "section": "executive"},
        {"KPI": "Lots quarantined / released",
         "value": f"{len(facts['lots'])} / {fmt_num(released, 0) if released is not None else '-'}",
         "section": "executive"},
        {"KPI": "Exposure remaining", "value": f"${fmt_num(exposure, 0)}", "section": "executive"},
    ])  # fmt: skip
    blocks.append(data_source_note(
        "control-plane Lakebase (alert episodes, signed decisions) + UC Delta (lot volume)",
        ["alert / alert_event / alert_response", PIPE.display()], WINDOW_TEXT))  # fmt: skip
    emit_html("".join(blocks))


executive_section()

## EXECUTIVE: cost of poor quality and lot exposure

For <b>Alex Morgan, VP Quality &amp; Site Head</b>. Every Spindle 3 excursion of the 60 °C validated limit puts the parts cut during it at risk. Containing them early (quarantine, CMM, disposition) costs an inspection; missing them costs the lot.

Cost of poor quality avoided

$162,000

9 excursions x 80% capture x ($24,000 - $1,500)

Lots quarantined

1

2 parts held, 13 released from the same lots

Lots released

11

of 12 lots machined in the window

Exposure remaining

$264,000

11 excursion lots without a quarantine decision

table view

| KPI | value | detail |
|---|---|---|
| Cost of poor quality avoided | $162,000 | 9 excursions x 80% capture x ($24,000 - $1,500) |
| Lots quarantined | 1 | 2 parts held, 13 released from the same lots |
| Lots released | 11 | of 12 lots machined in the window |
| Exposure remaining | $264,000 | 11 excursion lots without a quarantine decision |

<b>Containment realised:</b> 1 of 9 excursions (11%) carry a signed quarantine decision. The value model applies the capture-rate assumption to <i>every</i> detected excursion (what detection makes possible); exposure remaining is the value still at risk until each excursion lot is dispositioned.

#### Sensitivity of the value model

| capture rate | COPQ avoided |
|---|---|
| 50% | $101,250 |
| 70% | $141,750 |
| 80% | $162,000 |
| 90% | $182,250 |
| 100% | $202,500 |

| item | value |
|---|---|
| formula | events x capture rate x (lot cost - inspection cost) |
| events | 9 Spindle 3 excursion episodes (alert rule mfg-alert-spindle-temp) opened |
| lot cost / inspection cost | illustrative defaults; set the widgets to site values |
| lots quarantined | distinct lot_ids recorded by signed quarantine_lot decisions |
| lots released | lots machined (distinct lot_id on cnc_1_* readings) minus quarantined |
| exposure remaining | lots cut during excursions with no quarantine decision x lot cost |

#### Data source

| item | value |
|---|---|
| tier | control-plane Lakebase (alert episodes, signed decisions) + UC Delta (lot volume) |
| table | alert / alert_event / alert_response, <catalog>.<schema>.lb__shadow_manufacturing_sensor_readings_history |
| time window | last 2 h (2026-11-20 14:21 UTC to 2026-11-20 16:21 UTC) |

## OPERATIONS (Shreya, Head of Manufacturing Operations)

Detection and response times for Spindle 3 excursions, the quality actions they triggered, projected right-first-time, recovery time and the end-to-end latency of the stream that feeds it all.

In [ ]:
LATENCY_EDGES_MS = (0, 25, 50, 100, 250, 500, 1000, 5000)


def _stat(values: list[float | None], label: str, what: str) -> tuple[str, str, str]:
    data = [v for v in values if v is not None]
    if not data:
        return (label, "-", f"{what}: no data in the window")
    return (label, fmt_seconds(percentile(data, 0.5)),
            f"median of {len(data)}; p90 {fmt_seconds(percentile(data, 0.9))}")  # fmt: skip


def operations_section() -> None:
    blocks = [heading("OPERATIONS: detect, respond, recover", 2),
              paragraph("For <b>Shreya, Head of Manufacturing Operations</b>. How fast the cell "
                        "sees an excursion, how fast people act on it, what it triggered, and "
                        "how long the stream itself takes.")]  # fmt: skip
    try:
        episodes = _spindle(_episodes())
        responses = _responses()
    except Exception as exc:
        blocks.append(not_available("Operations KPIs", failure_text(exc)))
        emit_html("".join(blocks))
        return
    try:
        onsets = [r["onset"] for r in DATA.cold("excursions")]
    except Exception as exc:
        onsets = []
        blocks.append(note(f"Breach onsets from telemetry: {failure_text(exc)}", "NA"))
    ttd = match_onsets(episodes, onsets)
    tta = [seconds_between(ep["opened_at"], ep["acknowledged_at"]) for ep in episodes]
    mttr = [seconds_between(ep["opened_at"], ep["cleared_at"]) for ep in episodes]
    actions = {a: sum(1 for r in responses if r.get("action") == a) for a in QUALITY_ACTIONS}
    rft_rows = []
    for ep in episodes:
        end = ep.get("cleared_at") or RUN_STARTED
        rft_rows.append(projected_rft(ep["opened_at"], end, end))
    rft = projected_rft_total(rft_rows)
    acked = sum(1 for v in tta if v is not None)
    blocks.append(stat_tiles([
        _stat(ttd, "Time to detect", "first breach second to alert opened"),
        _stat(tta, "Time to acknowledge", f"{acked} of {len(episodes)} episodes acknowledged"),
        ("NCRs / CMM inspections", f"{actions['open_ncr']} / {actions['request_cmm_inspection']}",
         f"{actions['quarantine_lot']} quarantines, {actions['request_maintenance']} maintenance"),
        (PROJECTED_RFT_LABEL, fmt_pct(rft["rft"], 1) if rft["rft"] is not None else "-",
         f"{fmt_num(rft['good'], 0)} of {fmt_num(rft['parts_started'], 0)} parts started in "
         "excursion lots were cut inside the limit"),
        _stat(mttr, "MTTR", "alert opened to cleared (back inside the validated limit)"),
    ]))  # fmt: skip
    rows = [{"opened": fmt_ts(ep["opened_at"]), "detect": fmt_seconds(d), "acknowledge": fmt_seconds(a),
             "recover": fmt_seconds(m)}
            for ep, d, a, m in list(zip(episodes, ttd, tta, mttr, strict=True))[-12:]]  # fmt: skip
    if rows:
        blocks.append(heading("Latest Spindle 3 episodes", 4) + html_table(rows))
    blocks.append(_latency_block())
    SUMMARY.extend([
        {"KPI": "Time to detect (median)", "value": fmt_seconds(percentile([v for v in ttd if v is not None], 0.5)), "section": "operations"},
        {"KPI": "Time to acknowledge (median)", "value": fmt_seconds(percentile([v for v in tta if v is not None], 0.5)), "section": "operations"},
        {"KPI": "NCRs / inspections", "value": f"{actions['open_ncr']} / {actions['request_cmm_inspection']}", "section": "operations"},
        {"KPI": PROJECTED_RFT_LABEL, "value": fmt_pct(rft["rft"], 1) if rft["rft"] is not None else "-", "section": "operations"},
        {"KPI": "MTTR (median)", "value": fmt_seconds(percentile([v for v in mttr if v is not None], 0.5)), "section": "operations"},
    ])  # fmt: skip
    blocks.append(kv_table([
        ("time to detect", "alert opened minus the first second the raw telemetry crossed 60 °C"),
        ("time to acknowledge", "acknowledged event minus opened event of the same episode"),
        ("NCRs / inspections", "signed open_ncr / request_cmm_inspection decisions"),
        (PROJECTED_RFT_LABEL.lower(), PROJECTED_RFT_DEFINITION),
        ("MTTR", "cleared event minus opened event (temperature back inside the limit)"),
    ]))  # fmt: skip
    blocks.append(data_source_note(
        "control-plane Lakebase + UC Delta + streaming metrics table",
        ["alert / alert_event / alert_response", PIPE.display(), PIPE.display(METRICS_TABLE)],
        WINDOW_TEXT))  # fmt: skip
    emit_html("".join(blocks))


def _latency_block() -> str:
    out = heading("Streaming latency distribution", 4)
    try:
        s = DATA.latency("latency_summary")[0]
        batches = DATA.latency("latency_batches")
    except Exception as exc:
        return out + note(f"Streaming metrics: {failure_text(exc)}", "NA")
    if not num(s.get("batches")):
        return out + note("No streaming batches recorded in the window", "NA")
    out += stat_tiles([
        ("e2e p50 (median batch)", fmt_ms(s.get("p50_of_p50")), "median of per-batch e2e p50"),
        ("e2e p99 (median batch)", fmt_ms(s.get("p50_of_p99")), "median of per-batch e2e p99"),
        ("e2e p99 (tail batches)", fmt_ms(s.get("p99_of_p99")), "99th percentile of per-batch p99"),
        ("worst batch p99", fmt_ms(s.get("max_p99")), f"{fmt_num(s.get('batches'), 0)} batches"),
    ])  # fmt: skip
    hist = latency_histogram([b.get("e2e_p99") for b in batches], LATENCY_EDGES_MS)
    out += column_chart(hist, title="Per-batch e2e p99 latency", y_label="batches",
                        desc="Count of streaming batches by their e2e p99 latency bucket")  # fmt: skip
    out += paragraph(f"Window: {fmt_ts(s.get('first_batch'))} to {fmt_ts(s.get('last_batch'))} "
                     f"({fmt_num(s.get('batches'), 0)} batches, mode {PARAMS.mode}). Percentiles "
                     "are across batches of the per-batch values Spark reports.")  # fmt: skip
    return out


operations_section()

## OPERATIONS: detect, respond, recover

For <b>Shreya, Head of Manufacturing Operations</b>. How fast the cell sees an excursion, how fast people act on it, what it triggered, and how long the stream itself takes.

Time to detect

11s

median of 6; p90 2.5 min

Time to acknowledge

11s

median of 1; p90 11s

NCRs / CMM inspections

1 / 1

1 quarantines, 0 maintenance

Projected right-first-time

51.7%

74 of 143 parts started in excursion lots were cut inside the limit

MTTR

1.6 min

median of 9; p90 4.5 min

table view

| KPI | value | detail |
|---|---|---|
| Time to detect | 11s | median of 6; p90 2.5 min |
| Time to acknowledge | 11s | median of 1; p90 11s |
| NCRs / CMM inspections | 1 / 1 | 1 quarantines, 0 maintenance |
| Projected right-first-time | 51.7% | 74 of 143 parts started in excursion lots were cut inside the limit |
| MTTR | 1.6 min | median of 9; p90 4.5 min |

#### Latest Spindle 3 episodes

| opened | detect | acknowledge | recover |
|---|---|---|---|
| 2026-11-20 15:11 UTC | 2.7 min | - | 3.8 min |
| 2026-11-20 15:22 UTC | - | - | 1.9 min |
| 2026-11-20 15:31 UTC | - | - | 65s |
| 2026-11-20 15:39 UTC | - | - | 1.6 min |
| 2026-11-20 15:56 UTC | 2.4 min | - | 7.1 min |
| 2026-11-20 16:03 UTC | 7s | - | 56s |
| 2026-11-20 16:08 UTC | 10s | 11s | 1.7 min |
| 2026-11-20 16:13 UTC | 10s | - | 53s |
| 2026-11-20 16:18 UTC | 11s | - | 53s |

#### Streaming latency distribution

e2e p50 (median batch)

7 ms

median of per-batch e2e p50

e2e p99 (median batch)

23 ms

median of per-batch e2e p99

e2e p99 (tail batches)

304.5 s

99th percentile of per-batch p99

worst batch p99

317.1 s

5 batches

table view

| KPI | value | detail |
|---|---|---|
| e2e p50 (median batch) | 7 ms | median of per-batch e2e p50 |
| e2e p99 (median batch) | 23 ms | median of per-batch e2e p99 |
| e2e p99 (tail batches) | 304.5 s | 99th percentile of per-batch p99 |
| worst batch p99 | 317.1 s | 5 batches |

[chart] Per-batch e2e p99 latency: Count of streaming batches by their e2e p99 latency bucket

Window: 2026-11-20 15:15 UTC to 2026-11-20 16:16 UTC (5 batches, mode realtime). Percentiles are across batches of the per-batch values Spark reports.

| item | value |
|---|---|
| time to detect | alert opened minus the first second the raw telemetry crossed 60 °C |
| time to acknowledge | acknowledged event minus opened event of the same episode |
| NCRs / inspections | signed open_ncr / request_cmm_inspection decisions |
| projected right-first-time | Projected right-first-time = (parts started - parts at risk) / parts started, over the lots an excursion touched. Parts at risk are the parts whose cut overlapped the time Spindle 3 was outside its validated limit. Projected: no CMM results exist yet. |
| MTTR | cleared event minus opened event (temperature back inside the limit) |

#### Data source

| item | value |
|---|---|
| tier | control-plane Lakebase + UC Delta + streaming metrics table |
| table | alert / alert_event / alert_response, <catalog>.<schema>.lb__shadow_manufacturing_sensor_readings_history, <catalog>.<schema>.manufacturing_metrics |
| time window | last 2 h (2026-11-20 14:21 UTC to 2026-11-20 16:21 UTC) |

## Model risk score and AI action plans (placeholders)

These two sections are reserved for features that are being built in separate pull requests (a Spindle 3 thermal-risk model served from Unity Catalog, and grounded GenAI action plans). Until they land, the notebook says so instead of showing numbers.

In [ ]:
PENDING_FEATURES = (
    ("Model risk score",
     "Spindle 3 thermal-risk score per part from the UC model spindle_thermal_risk served by the "
     "endpoint pharma-twin-spindle-risk. Pending: lands with the ML risk-model pull request."),
    ("AI action plans",
     "Grounded GenAI action plans for each excursion (what to quarantine, inspect and fix). "
     "Pending: lands with the GenAI action-plan pull request."),
)  # fmt: skip


def pending_section() -> None:
    blocks = [heading("Coming next (not yet available)", 2)]
    for title, text in PENDING_FEATURES:
        blocks.append(heading(title, 4) + note(f"PENDING: {text}", "NA"))
    emit_html("".join(blocks))


pending_section()

## Coming next (not yet available)

#### Model risk score

PENDING: Spindle 3 thermal-risk score per part from the UC model spindle_thermal_risk served by the endpoint pharma-twin-spindle-risk. Pending: lands with the ML risk-model pull request.

#### AI action plans

PENDING: Grounded GenAI action plans for each excursion (what to quarantine, inspect and fix). Pending: lands with the GenAI action-plan pull request.

## Summary

The headline numbers in one table, plus which data sources answered.

In [ ]:
def summary_section() -> None:
    sources = [{"data source": k, "state": v} for k, v in sorted(DATA.sources.items())]
    finished = datetime.now(UTC)
    emit_html(
        heading("Summary", 2)
        + sample_banner()
        + html_table(SUMMARY, ["KPI", "value", "section"])
        + heading("Data sources", 4)
        + html_table(sources, ["data source", "state"])
        + paragraph(f"Run finished {fmt_ts(finished)}; window {WINDOW_TEXT}.")
    )
    if BACKEND is not None:
        BACKEND.close()


summary_section()

## Summary

| KPI | value | section |
|---|---|---|
| Readings analysed | 2,607,155 | glance |
| COPQ avoided (model) | $162,000 | executive |
| Lots quarantined / released | 1 / 11 | executive |
| Exposure remaining | $264,000 | executive |
| Time to detect (median) | 11s | operations |
| Time to acknowledge (median) | 11s | operations |
| NCRs / inspections | 1 / 1 | operations |
| Projected right-first-time | 51.7% | operations |
| MTTR (median) | 1.6 min | operations |

#### Data sources

| data source | state |
|---|---|
| Lakebase LakeTS (hot tier) | reached |
| UC Delta (cold tier) | reached |
| control-plane Lakebase (alerts, decisions) | reached |
| streaming metrics table | reached |

Run finished 2026-11-20 16:21 UTC; window last 2 h (2026-11-20 14:21 UTC to 2026-11-20 16:21 UTC).